# General-Purpose Classical Reasoning Agent

End-to-end system for the AI assignment: **Natural Language Knowledge Base → Internal Representation → Classical Reasoning → True/False**.

Pipeline:
- **Router** (Part 4) sends each item to one of three engines.
- **Part 1 — RuleTaker engine**: NL facts/rules → first-order Horn clauses → forward-chaining least-fixpoint closure with negation-as-failure and contradiction retraction.
- **Part 2 — CounterBench engine**: NL causal mechanisms → structural causal model → Pearl's abduction–action–prediction counterfactual procedure (exhaustive root search + `do()` + fixed-point propagation).
- **Part 3 — FOLIO engine**: NL/FOL → FOL → NNF → Skolem CNF → resolution refutation (unification, factoring, subsumption, set-of-support).
- Final decision always comes from the classical engine (generative AI is never consulted).

Run the cells top to bottom: setup → Parts 1–4 → smoke tests and traces → full evaluation (prints final accuracy and inference time) → `submission.csv` generation.

In [1]:
"""Setup: standard imports and the in-notebook module loader.

The complete reasoning system (Parts 1-4 below) is embedded in this
notebook as source cells and materialised into real Python modules here,
so cross-module imports work exactly as in the developed package.
No file outside this notebook is required to run the agent itself.
"""
import json
import os
import re
import sys
import time
import types
import itertools
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, FrozenSet, List, Optional, Sequence, Set, Tuple


def _mod(name, src):
    mod = types.ModuleType(name)
    mod.__dict__["__name__"] = name
    sys.modules[name] = mod
    exec(compile(src, name + ".py", "exec"), mod.__dict__)
    return mod


print("setup ok: python", sys.version.split()[0])


setup ok: python 3.13.17


In [2]:
# Part 1 — RuleTaker engine: NL → Horn clauses → forward chaining (least fixpoint) + backward chaining for traces.
ruletaker = _mod('ruletaker', r'''"""
RuleTaker engine.

Compiles RuleTaker-style natural-language knowledge bases

    facts : "Anne is big."  /  "The cat does not chase the cow."
    rules : "If someone is young and kind then they are smart."
            "All furry things are round."  /  "Kind, young things are not smart."

into an internal first-order Horn representation

    facts : ground literals        Quiet(bob), not Young(anne)
    rules : body literals -> head  [Young(x), Kind(x)] -> Successful(x)

and answers queries by FORWARD CHAINING (least-fixpoint closure, the classic
iterative deduction algorithm) with NEGATION AS A FIRST-CLASS LITERAL
(explicit negative facts/heads, no negation-as-failure).  A query is answered
True iff its literal is in the closure.  A backward-chaining engine (DFS over
the AND-OR proof graph with unification) is also provided; it produces the
explicit reasoning traces used in the report.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Dict, FrozenSet, Iterable, List, Optional, Sequence, Set, Tuple

# ---------------------------------------------------------------------------
# Internal representation
#
# Term  : ('c', name) constant | ('v', name) variable
# Literal: (predicate, args, positive?)   e.g. ('Successful', (('c','Bob'),), True)
# Rule  : conjunction of literals -> conjunction of literals (universally
#         quantified over the variables appearing in it)
# ---------------------------------------------------------------------------

Term = Tuple[str, str]
Lit = Tuple[str, Tuple[Term, ...], bool]


def const(name: str) -> Term:
    return ("c", name)


def var(name: str) -> Term:
    return ("v", name)


def render_term(t: Term) -> str:
    return t[1] if t[0] == "c" else f"?{t[1]}"


def render_lit(lit: Lit) -> str:
    pred, args, pos = lit
    inner = ", ".join(render_term(a) for a in args)
    return f"{'not ' if not pos else ''}{pred}({inner})"


@dataclass
class Rule:
    body: Tuple[Lit, ...]
    head: Tuple[Lit, ...]
    text: str = ""

    def render(self) -> str:
        body = " and ".join(render_lit(l) for l in self.body)
        head = " and ".join(render_lit(l) for l in self.head)
        return f"{body} -> {head}"


@dataclass
class KnowledgeBase:
    facts: List[Lit] = field(default_factory=list)
    rules: List[Rule] = field(default_factory=list)
    constants: List[str] = field(default_factory=list)
    raw_sentences: List[str] = field(default_factory=list)

    @property
    def constant_terms(self) -> List[Term]:
        return [const(c) for c in self.constants]


class ParseError(ValueError):
    pass


# ---------------------------------------------------------------------------
# Natural-language parsing
# ---------------------------------------------------------------------------

_ADJ = r"[a-z]+"
_VERB_BASES = {
    "likes": "like", "chases": "chase", "sees": "see", "needs": "need",
    "visits": "visit", "eats": "eat", "like": "like", "chase": "chase",
    "see": "see", "need": "need", "visit": "visit", "eat": "eat",
}

_VAR_NPS = {"someone", "something", "everything", "everyone",
            "they", "them", "it"}


def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())


def _pred(name: str) -> str:
    return name[0].upper() + name[1:]


def _entity_np(np: str) -> str:
    """Normalize an entity noun phrase.

    - 'the bald eagle' / 'the cat' -> 'bald_eagle' / 'cat' (full phrase kept,
      so 'bald eagle' and 'eagle' never collide);
    - person names ('Anne') are kept as-is.
    """
    np = np.strip()
    if np.lower().startswith("the "):
        content = np[4:].strip()
        return "_".join(content.lower().split())
    return np.strip()


def _adj_list(txt: str) -> List[str]:
    """'blue, quiet' / 'furry' / 'young and kind' -> list of adjectives."""
    parts = re.split(r",\s*|\s+and\s+", txt.strip())
    return [p.strip() for p in parts if p.strip()]


class _RuleParser:
    """Parses one rule body/head, threading the subject variable."""

    def __init__(self):
        self.varname = "x"

    def var(self) -> Term:
        return var(self.varname)

    def _check_np(self, np: str) -> None:
        """Validate that a string is a well-formed entity noun phrase."""
        low = np.strip().lower()
        if low in _VAR_NPS:
            return
        if low.startswith("the "):
            rest = np.strip()[4:].strip()
            if rest and re.fullmatch(r"[A-Za-z]+(?:\s+[A-Za-z]+)*", rest):
                return
            raise ParseError(f"bad entity phrase: {np!r}")
        if re.fullmatch(r"[A-Z][a-z]*", np.strip()):
            return
        raise ParseError(f"bad entity phrase: {np!r}")

    def subject_term(self, subj: str) -> Term:
        low = subj.strip().lower()
        if low in _VAR_NPS:
            return self.var()
        self._check_np(subj)
        return const(_entity_np(subj))

    def object_term(self, obj: str) -> Term:
        low = obj.strip().lower()
        if low in _VAR_NPS:
            return self.var()
        self._check_np(obj)
        return const(_entity_np(obj))

    def parse_clause(self, clause: str) -> Lit:
        """Parse one simple clause into a literal.

        Clause shapes (subject boundary is resolved through the closed
        verb lexicon, so multi-word entities such as 'the bald eagle'
        are never mis-split):
          '<subj> is|are [not] <adj>'
          '<subj> does|do not <verb-base> <obj>'
          '<subj> <verb> <obj>'
        Trailing commas ('young, then ...') are tolerated.
        """
        clause = _norm(clause).strip().rstrip(".").strip()
        clause = re.sub(r",\s*$", "", clause).strip()
        if not clause:
            raise ParseError("empty clause")
        # --- unary: '<subj> is|are [not] <adj>' ---
        m = re.fullmatch(rf"(.+?)\s+(is|are)\s+(not\s+)?([A-Za-z]+)", clause)
        if m:
            subj, adj = m.group(1).strip(), m.group(4).lower()
            try:
                subj_t = self.subject_term(subj)
            except ParseError:
                subj_t = None
            if subj_t is not None and adj not in _VERB_BASES:
                pos = not m.group(3)
                return (_pred(adj), (subj_t,), pos)
        # --- binary with negation: '<subj> does|do not <verb> <obj>' ---
        m = re.fullmatch(rf"(.+?)\s+(?:does|do)\s+not\s+([A-Za-z]+)\s+(.+)",
                         clause)
        if m:
            subj, verb, obj = (m.group(1).strip(), m.group(2).lower(),
                               m.group(3).strip())
            base = _VERB_BASES.get(verb)
            if base is not None:
                return (_pred(base),
                        (self.subject_term(subj), self.object_term(obj)), False)
            raise ParseError(f"unknown verb {verb!r} in {clause!r}")
        # --- binary positive: locate the verb token from the lexicon ---
        toks = clause.split()
        vidx = next((i for i, t in enumerate(toks)
                     if t.lower() in _VERB_BASES), None)
        if vidx is not None and vidx > 0 and vidx < len(toks) - 1:
            subj = " ".join(toks[:vidx])
            obj = " ".join(toks[vidx + 1:])
            base = _VERB_BASES[toks[vidx].lower()]
            return (_pred(base),
                    (self.subject_term(subj), self.object_term(obj)), True)
        raise ParseError(f"unparsable clause: {clause!r}")

    def parse_conjunction(self, txt: str) -> List[Lit]:
        """Split on ' and ' and parse each clause.

        Handles elliptical continuations that share the previous subject,
        e.g. 'someone is white and rough' -> 'someone is white',
        'someone is rough'; 'something is round and not smart' likewise.
        Commas also separate conjuncts ('smart and kind,').
        """
        parts = re.split(r"\s+and\s+|,\s*", txt)
        out: List[Lit] = []
        last_subj: Optional[str] = None
        for p in parts:
            p = p.strip().rstrip(".").strip()
            if not p:
                continue
            try:
                lit = self.parse_clause(p)
                out.append(lit)
                last_subj = self._clause_subject(p)
            except ParseError as e:
                if last_subj is not None and re.fullmatch(
                        rf"(not\s+)?[A-Za-z]+", p):
                    lit = self.parse_clause(f"{last_subj} is {p}")
                    out.append(lit)
                else:
                    raise
        if not out:
            raise ParseError(f"empty conjunction: {txt!r}")
        return out

    @staticmethod
    def _clause_subject(clause: str) -> Optional[str]:
        """Recover the raw subject text of an already-parsable clause."""
        c = _norm(clause).strip()
        m = re.fullmatch(rf"(.+?)\s+(is|are)\s+(not\s+)?([A-Za-z]+)", c)
        if m:
            return m.group(1).strip()
        m = re.fullmatch(rf"(.+?)\s+(?:does|do)\s+not\s+([A-Za-z]+)\s+(.+)", c)
        if m:
            return m.group(1).strip()
        toks = c.split()
        vidx = next((i for i, t in enumerate(toks)
                     if t.lower() in _VERB_BASES), None)
        if vidx is not None and vidx > 0 and vidx < len(toks) - 1:
            return " ".join(toks[:vidx])
        return None


def _split_top(s: str, seps: Sequence[str]) -> List[str]:
    """Split on the first separator that yields >1 part (for 'If A then B')."""
    for sep in seps:
        if sep in s:
            i = s.index(sep)
            return [s[:i], s[i + len(sep):]]
    return [s]


def parse_sentence(sent: str) -> Tuple[Optional[Lit], Optional[Rule]]:
    """Parse one sentence; returns (fact, rule) with one of them None."""
    s = _norm(sent).strip().rstrip(".").strip()
    if not s:
        return None, None

    low = s.lower()
    # ---- universal 'All ...' / '<adj-list> things/people are ...' rules ----
    m = re.fullmatch(
        rf"(?:all\s+)?((?:{_ADJ}[,\s]+)*{_ADJ})\s+(?:things|people|animals)\s+"
        rf"are\s+(not\s+)?({_ADJ})",
        s,
        re.I,
    )
    if m:
        adjs = _adj_list(m.group(1))
        pos = not m.group(2)
        head_adj = m.group(3).lower()
        body = [(_pred(a), (var("x"),), True) for a in adjs]
        head = [(_pred(head_adj), (var("x"),), pos)]
        return None, Rule(tuple(body), tuple(head), s)

    # ---- conditional rules ----
    if low.startswith("if "):
        rest = s[3:]
        split = _split_top(rest, [" then "])
        if len(split) != 2:
            raise ParseError(f"rule without 'then': {s!r}")
        body_txt, head_txt = split
        parser = _RuleParser()
        body = parser.parse_conjunction(body_txt)
        head = parser.parse_conjunction(head_txt)
        return None, Rule(tuple(body), tuple(head), s)

    # ---- facts ----
    parser = _RuleParser()
    lit = parser.parse_clause(s)
    return lit, None


def parse_context(context: str) -> KnowledgeBase:
    """Parse a RuleTaker context (facts + rules) into a KnowledgeBase."""
    kb = KnowledgeBase()
    constants: List[str] = []

    def add_constant(term: Term):
        if term[0] == "c" and term[1] not in constants:
            constants.append(term[1])

    for sent in re.split(r"\.(?=\s|$)", context):
        sent = sent.strip()
        if not sent:
            continue
        kb.raw_sentences.append(sent)
        fact, rule = parse_sentence(sent)
        if fact is not None:
            kb.facts.append(fact)
            for a in fact[1]:
                add_constant(a)
        elif rule is not None:
            kb.rules.append(rule)
            for lit in rule.body + rule.head:
                for a in lit[1]:
                    add_constant(a)
        else:
            raise ParseError(f"empty sentence in context: {sent!r}")
    kb.constants = constants
    return kb


def parse_query(statement: str) -> Lit:
    """Parse a query statement (same grammar as facts)."""
    fact, _ = parse_sentence(statement)
    if fact is None:
        raise ParseError(f"unparsable query: {statement!r}")
    return fact


# ---------------------------------------------------------------------------
# Unification
# ---------------------------------------------------------------------------


def unify(pattern_args: Tuple[Term, ...], fact_args: Tuple[Term, ...],
          subst: Dict[str, Term]) -> Optional[Dict[str, Term]]:
    """Standard first-order unification of argument tuples (patterns may
    contain variables; facts are ground)."""
    for p, f in zip(pattern_args, fact_args):
        if p[0] == "v":
            bound = subst.get(p[1])
            if bound is None:
                subst[p[1]] = f
            elif bound != f:
                return None
        else:
            if p != f:
                return None
    return subst


def substitute(args: Tuple[Term, ...], subst: Dict[str, Term]) -> Tuple[Term, ...]:
    return tuple(subst.get(a[1], a) if a[0] == "v" else a for a in args)


# ---------------------------------------------------------------------------
# Forward chaining (least-fixpoint closure) - the primary reasoning engine
# ---------------------------------------------------------------------------


@dataclass
class Closure:
    facts: FrozenSet[Lit]
    trace: Dict[Lit, Tuple[str, Tuple[Lit, ...]]] = field(default_factory=dict)


def forward_chain(kb: KnowledgeBase, max_rounds: int = 200) -> Closure:
    """Compute the least-fixpoint closure of the facts under the rules.

    Classical iterated forward chaining (a.k.a. bottom-up / data-driven
    deduction).  Both positive and negative literals are derived; the KB may
    be inconsistent (both P and not-P derivable), which is fine - the closure
    simply records whatever is derivable.
    """
    facts: Set[Lit] = set(kb.facts)
    # index rules by a body predicate for quick matching
    rule_index: Dict[Tuple[str, bool], List[int]] = {}
    for i, r in enumerate(kb.rules):
        for lit in r.body:
            key = (lit[0], lit[2])
            rule_index.setdefault(key, []).append(i)
    fact_index: Dict[Tuple[str, bool], List[Lit]] = {}
    for f in facts:
        fact_index.setdefault((f[0], f[2]), []).append(f)
    trace: Dict[Lit, Tuple[str, Tuple[Lit, ...]]] = {
        f: ("given", (f,)) for f in facts
    }

    changed = True
    rounds = 0
    while changed and rounds < max_rounds:
        changed = False
        rounds += 1
        for i, rule in enumerate(kb.rules):
            # enumerate all substitutions satisfying the body, then add heads
            subs = [{}, ]
            ok = True
            for lit in rule.body:
                nxt = []
                key = (lit[0], lit[2])
                for sub in subs:
                    for f in fact_index.get(key, ()):
                        s2 = unify(lit[1], f[1], dict(sub))
                        if s2 is not None:
                            nxt.append(s2)
                subs = nxt
                if not subs:
                    ok = False
                    break
            if not ok:
                continue
            for sub in subs:
                for h in rule.head:
                    new = (h[0], substitute(h[1], sub), h[2])
                    if new not in facts:
                        facts.add(new)
                        fact_index.setdefault((new[0], new[2]), []).append(new)
                        changed = True
                        trace[new] = ("rule", tuple(
                            (l[0], substitute(l[1], sub), l[2]) for l in rule.body
                        ))
    return Closure(facts=frozenset(facts), trace=trace)


# ---------------------------------------------------------------------------
# Backward chaining (DFS with unification) - used for reasoning traces
# ---------------------------------------------------------------------------


def backward_chain(query: Lit, kb: KnowledgeBase,
                   max_depth: int = 30) -> Optional[List[Lit]]:
    """Depth-first AND/OR search for a proof of a ground query literal.

    Returns a list of derived literals forming the proof (the reasoning
    trace), or None if the query is not derivable.
    """
    stack: List[Tuple[Lit, int, List[Lit]]] = [(query, 0, [])]
    while stack:
        goal, depth, proof = stack.pop()
        if depth > max_depth:
            continue
        # fact?
        if goal in kb.facts:
            return proof + [goal]
        # rule head?
        for rule in kb.rules:
            for h in rule.head:
                if h[0] != goal[0] or h[2] != goal[2]:
                    continue
                sub = unify(h[1], goal[1], {})
                if sub is None:
                    continue
                ground_body = [
                    (l[0], substitute(l[1], sub), l[2]) for l in rule.body
                ]
                # AND-node: try to prove each body literal (simple DFS with
                # re-queueing is avoided by using a recursive helper below)
                ok, subproof = _prove_all(ground_body, kb, depth + 1, max_depth)
                if ok:
                    return proof + subproof + [goal]
        continue
    return None


def _prove_all(goals: List[Lit], kb: KnowledgeBase, depth: int,
               max_depth: int) -> Tuple[bool, List[Lit]]:
    if not goals:
        return True, []
    if depth > max_depth:
        return False, []
    goal, rest = goals[0], goals[1:]
    # fact?
    if goal in kb.facts:
        ok, proof = _prove_all(rest, kb, depth, max_depth)
        if ok:
            return True, [goal] + proof
        return False, []
    # rule head?
    for rule in kb.rules:
        for h in rule.head:
            if h[0] != goal[0] or h[2] != goal[2]:
                continue
            sub = unify(h[1], goal[1], {})
            if sub is None:
                continue
            ground_body = [
                (l[0], substitute(l[1], sub), l[2]) for l in rule.body
            ]
            ok, proof1 = _prove_all(ground_body, kb, depth + 1, max_depth)
            if not ok:
                continue
            ok, proof2 = _prove_all(rest, kb, depth, max_depth)
            if ok:
                return True, proof1 + [goal] + proof2
    return False, []


def _relevant_atoms(kb: KnowledgeBase) -> List[Lit]:
    """All positive ground literals over the KB's predicates and constants."""
    preds: Dict[str, int] = {}
    for f in kb.facts:
        preds.setdefault(f[0], len(f[1]))
    for r in kb.rules:
        for lit in r.body + r.head:
            preds.setdefault(lit[0], len(lit[1]))
    consts = [const(c) for c in kb.constants]
    atoms: List[Lit] = []
    for pred, arity in preds.items():
        if arity == 1:
            for c in consts:
                atoms.append((pred, (c,), True))
        elif arity == 2:
            for a in consts:
                for b in consts:
                    atoms.append((pred, (a, b), True))
    return atoms


def cwa_closure(kb: KnowledgeBase, max_rounds: int = 50) -> Closure:
    """Forward-chaining closure under negation-as-failure (CWA).

    Classical iterated forward chaining over literals, extended with the
    closed-world rule: any relevant positive ground atom that is not
    derivable is assumed false (its explicit negation is added) and may
    itself fire rules.  This matches the RuleTaker labeling convention,
    under which e.g. 'Bob is not big' counts as entailed whenever
    'Bob is big' is not derivable.

    The assumption set is recomputed from scratch after every chaining
    pass and iterated to a fixpoint: an assumption that turns out to be
    contradicted (its positive counterpart becomes derivable) is dropped,
    which can in turn unblock other assumptions (e.g. once
    'Big(bald_eagle)' is derived, 'Visit(bald_eagle, cow)' loses its only
    support, so 'not Visit(bald_eagle, cow)' -- and hence
    'Cold(bald_eagle)' -- become derivable again).  The result is the
    natural stratified reading of the theory.  Assumption sets are finite,
    so the loop either stabilises or repeats a previous set; in the rare
    cyclic case the most recent self-consistent closure is used.
    """
    base: Set[Lit] = set(kb.facts)
    atoms = _relevant_atoms(kb)

    def chain(extra: Set[Lit]) -> Closure:
        trial = KnowledgeBase(
            facts=list(base | extra), rules=kb.rules,
            constants=kb.constants, raw_sentences=kb.raw_sentences,
        )
        return forward_chain(trial)

    assumed: Set[Lit] = set()
    closure = chain(assumed)
    history: List[FrozenSet[Lit]] = []
    for _ in range(max_rounds):
        derived = set(closure.facts)
        # drop contradicted assumptions, keep the rest, and assume false
        # anything still underivable
        contradicted = {
            a for a in assumed if (a[0], a[1], True) in derived
        }
        keep = assumed - contradicted
        add = {
            (atom[0], atom[1], False)
            for atom in atoms
            if (atom[0], atom[1], True) not in derived
            and (atom[0], atom[1], False) not in derived
        }
        new = keep | add
        if new == assumed:
            return closure  # fixpoint: stable and self-consistent
        if frozenset(new) in history:
            # genuine cycle: fall back to the classical closure augmented
            # with only the uncontradicted assumptions
            safe = {
                a for a in new
                if (a[0], a[1], True) not in derived
            }
            return chain(safe)
        history.append(frozenset(assumed))
        assumed = new
        closure = chain(assumed)
    return closure


def answer_cwa(context: str, statement: str) -> bool:
    """Predict True iff the statement holds under the CWA closure.

    Positive statements must be derived; negative statements hold exactly
    when their positive counterpart is not derived.
    """
    kb = parse_context(context)
    q = parse_query(statement)
    closure = cwa_closure(kb)
    if q[2]:
        return q in closure.facts
    return (q[0], q[1], True) not in closure.facts


def answer_batch_cwa(context: str, statements: Sequence[str]) -> List[bool]:
    """Answer many queries for one context with a single CWA closure."""
    kb = parse_context(context)
    closure = cwa_closure(kb)
    out = []
    for s in statements:
        q = parse_query(s)
        if q[2]:
            out.append(q in closure.facts)
        else:
            out.append((q[0], q[1], True) not in closure.facts)
    return out


def answer(context: str, statement: str) -> bool:
    """Predict True iff the statement is entailed by the context.

    Uses the CWA closure (negation as failure), matching the entailment
    convention of the RuleTaker labeling.
    """
    return answer_cwa(context, statement)


def answer_batch(context: str, statements: Sequence[str]) -> List[bool]:
    """Answer many queries for one context with a single closure computation."""
    return answer_batch_cwa(context, statements)

''')

In [3]:
# Part 2 — CounterBench engine: NL → structural causal model → abduction / action / prediction.
counterbench = _mod('counterbench', r'''"""
CounterBench engine.

Parses deterministic counterfactual causal questions of the form

    given_info: "We know that X causes Y, A and B together cause C, ...
                [P~ Bern(q).] [We observed [not] V ...]"
    question : "Would Y occur if not X instead of X?"        (basic/conditional/backdoor)
               "Would Y occur if not X1 and not X2?"          (joint)
               "Assume not A, and based on this assumption,
                further suppose not B. Would Y occur?"       (nested)

The natural language is compiled into an explicit structural causal model
(SCM) and the query is answered with the classical three-step counterfactual
procedure (Pearl):

    1. ABDUCTION   - search over the finite space of root (exogenous)
                     assignments, keeping those consistent with the
                     observations stated in the context;
    2. ACTION     - apply the do() interventions stated in the question;
    3. PREDICTION - propagate the deterministic structural equations
                     (forward chaining on the causal DAG, including
                     fixed-point search for cyclic dependency systems)
                     and read the query variable.

Everything is symbolic / classical: DAG construction, constraint
propagation, fixed-point search and exhaustive search.  No machine learning.
"""
from __future__ import annotations

import itertools
import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

_WORD = r"[A-Za-z][A-Za-z0-9]*"


class ParseError(ValueError):
    pass


# ---------------------------------------------------------------------------
# Parsing helpers
# ---------------------------------------------------------------------------


def _parse_part(p: str) -> Tuple[bool, str]:
    """Parse one cause-side part: 'Ziklo' or 'not Vork' -> (positive?, var)."""
    p = p.strip()
    m = re.fullmatch(rf"not\s+({_WORD})", p)
    if m:
        return (False, m.group(1))
    m = re.fullmatch(rf"({_WORD})", p)
    if m:
        return (True, m.group(1))
    raise ParseError(f"unparsable cause part: {p!r}")


def _parse_subject(s: str) -> Tuple[str, List[bool], List[str]]:
    """Parse the cause-side expression of a mechanism clause.

    Returns (op, signs, parents):
      op  in {'id', 'and', 'or'}
      signs[i] is True iff parent i enters positively.
    Examples:
      'Ziklo'                         -> ('id',  [True], ['Ziklo'])
      'not Vork'                      -> ('id',  [False], ['Vork'])
      'Blaf and Ziklo together'       -> ('and', [True, True],  ['Blaf', 'Ziklo'])
      'Trune and not Blaf together'   -> ('and', [True, False], ['Trune', 'Blaf'])
      'Ziklo or Blaf'                 -> ('or',  [True, True],  ['Ziklo', 'Blaf'])
      'not Trune or Ziklo'            -> ('or',  [False, True], ['Trune', 'Ziklo'])
    """
    s = s.strip().rstrip(".").strip()
    if not s:
        raise ParseError("empty subject")
    m = re.fullmatch(rf"(?:not\s+)?({_WORD})", s)
    if m:
        neg = bool(re.match(r"not\s+", s))
        return ("id", [not neg], [m.group(1)])
    s = re.sub(r"\s+together$", "", s)  # 'A and B together' / 'A or B together'
    if " or " in s:
        parts = [p.strip() for p in s.split(" or ")]
        if len(parts) == 2 and all(parts):
            parsed = [_parse_part(p) for p in parts]
            return ("or", [pos for pos, _ in parsed], [v for _, v in parsed])
        raise ParseError(f"bad or-subject: {s!r}")
    if " and " in s:
        parts = [p.strip() for p in s.split(" and ")]
        parts = [p for p in parts if p]
        if len(parts) >= 2:
            parsed = [_parse_part(p) for p in parts]
            return ("and", [pos for pos, _ in parsed], [v for _, v in parsed])
        raise ParseError(f"bad and-subject: {s!r}")
    raise ParseError(f"unparsable subject: {s!r}")


def _parse_object_list(s: str) -> List[Tuple[bool, str]]:
    """Parse the effect-side list, e.g. 'Blaf, Trune and Vork' or 'not Vork'."""
    s = s.strip().rstrip(".").strip()
    if not s:
        raise ParseError("empty object list")
    out: List[Tuple[bool, str]] = []
    for item in re.split(r",\s*|\s+and\s+", s):
        item = item.strip()
        if not item:
            continue
        m = re.fullmatch(rf"not\s+({_WORD})", item)
        if m:
            out.append((True, m.group(1)))  # negated effect
        elif re.fullmatch(rf"{_WORD}", item):
            out.append((False, item))
        else:
            raise ParseError(f"unparsable object item: {item!r}")
    if not out:
        raise ParseError(f"empty object list: {s!r}")
    return out


def parse_mechanisms(text: str) -> List[Tuple[str, List[bool], List[str], List[Tuple[bool, str]]]]:
    """Parse the 'We know that ...' mechanism sentence into raw clauses.

    Returns a list of (op, signs, parents, objects) where objects is the list
    of (negated?, var) effects.  Anchors on the verbs ('causes'/'cause').  The
    text between two consecutive verbs is <object list of clause i> + ', ' +
    <subject of clause i+1>.  Subjects never contain commas, so the final
    comma-chunk of each inter-verb segment is the next subject.
    """
    body = text.strip()
    if body.lower().startswith("we know that "):
        body = body[len("we know that ") :]
    body = body.strip().rstrip(".")
    if not body:
        return []
    segs = re.split(r"\b(?:causes|cause)\b", body)
    if len(segs) < 2:
        raise ParseError(f"no mechanism verb in: {text!r}")
    pairs: List[Tuple[str, str]] = []
    subj = segs[0].strip()
    if not subj:
        raise ParseError(f"missing first subject in: {text!r}")
    for i in range(1, len(segs)):
        seg = segs[i].strip()
        if i < len(segs) - 1:
            chunks = [c.strip() for c in seg.split(",")]
            # clause joiner ', and ...' may prefix any chunk; variable names
            # are single words so a leading 'and ' is always a joiner.
            chunks = [re.sub(r"^and\s+", "", c, flags=re.IGNORECASE) for c in chunks]
            chunks = [c for c in chunks if c]
            if len(chunks) < 2:
                raise ParseError(f"cannot split {seg!r} in: {text!r}")
            obj = ", ".join(chunks[:-1])
            pairs.append((subj, obj))
            subj = chunks[-1]
        else:
            pairs.append((subj, seg))
    clauses = []
    for subj_txt, obj_txt in pairs:
        op, signs, parents = _parse_subject(subj_txt)
        objects = _parse_object_list(obj_txt)
        clauses.append((op, signs, parents, objects))
    return clauses


# ---------------------------------------------------------------------------
# Structural causal model
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class Equation:
    op: str  # 'id' | 'and' | 'or'
    signs: Tuple[bool, ...]  # input polarity of each parent
    parents: Tuple[str, ...]
    out_neg: bool = False  # '... cause not <var>'

    def evaluate(self, values: Dict[str, bool]) -> bool:
        pv = [bool(values[p]) == s for p, s in zip(self.parents, self.signs)]
        if self.op == "id":
            val = pv[0]
        elif self.op == "and":
            val = all(pv)
        elif self.op == "or":
            val = any(pv)
        else:
            raise ValueError(f"bad op {self.op}")
        return (not val) if self.out_neg else val

    def render(self, var: str) -> str:
        terms = [(p if s else f"not {p}") for p, s in zip(self.parents, self.signs)]
        if self.op == "id":
            lhs = terms[0]
        elif self.op == "and":
            lhs = " and ".join(terms)
        else:
            lhs = " or ".join(terms)
        return f"{var} := {'not ' if self.out_neg else ''}({lhs})"


@dataclass
class CausalModel:
    equations: Dict[str, Equation] = field(default_factory=dict)
    priors: Dict[str, float] = field(default_factory=dict)
    observed: Dict[str, bool] = field(default_factory=dict)
    variables: List[str] = field(default_factory=list)
    clauses: List = field(default_factory=list)

    @property
    def roots(self) -> List[str]:
        return [v for v in self.variables if v not in self.equations]

    def fixed_points(self, forced: Dict[str, bool]) -> List[Dict[str, bool]]:
        """Evaluate all variables given root/intervention values `forced`.

        Deterministic variables are computed by forward propagation; any
        variables left unresolved (cyclic dependencies) are settled by
        exhaustive fixed-point search (a classical finite search over their
        Boolean combinations, keeping assignments consistent with every
        structural equation).
        """
        values: Dict[str, bool] = dict(forced)
        changed = True
        while changed:
            changed = False
            for var, eq in self.equations.items():
                if var in values:
                    continue
                if all(p in values for p in eq.parents):
                    values[var] = eq.evaluate(values)
                    changed = True
        unresolved = [v for v in self.variables if v not in values]
        if not unresolved:
            return [values]
        if len(unresolved) > 10:
            raise ParseError(f"too many unresolved variables: {unresolved}")
        out = []
        for combo in itertools.product([False, True], repeat=len(unresolved)):
            vals = dict(values)
            vals.update(dict(zip(unresolved, combo)))
            ok = True
            for var in unresolved:
                eq = self.equations.get(var)
                if eq is None:
                    continue  # free root value
                if eq.evaluate(vals) != vals[var]:
                    ok = False
                    break
            if ok:
                out.append(vals)
        return out


# ---------------------------------------------------------------------------
# Context parsing
# ---------------------------------------------------------------------------


def parse_context(given_info: str) -> CausalModel:
    """Parse a CounterBench given_info string into a CausalModel (raw clauses)."""
    text = given_info.strip()
    m = re.match(r"(.*?)(?:\.|$)", text)
    mech_txt = m.group(1) if m else text
    clauses = parse_mechanisms(mech_txt)
    priors: Dict[str, float] = {}
    for pm in re.finditer(rf"({_WORD})\s*~\s*Bern\((\d+(?:\.\d+)?)\)", text):
        priors[pm.group(1)] = float(pm.group(2))
    observed: Dict[str, bool] = {}
    for om in re.finditer(r"We observed ([^.]*?)(?=\.|$)", text):
        for item in re.split(r",\s*|\s+and\s+", om.group(1).strip()):
            item = item.strip()
            if not item:
                continue
            neg = False
            m2 = re.fullmatch(rf"not\s+({_WORD})", item)
            if m2:
                observed[m2.group(1)] = False
                continue
            m2 = re.fullmatch(rf"({_WORD})(.*)", item)
            if not m2:
                raise ParseError(f"unparsable observation: {item!r}")
            var, tail = m2.group(1), m2.group(2).strip()
            if tail:
                mt = re.fullmatch(r"(not\s+)?(?:occur|occurs)", tail)
                if not mt:
                    raise ParseError(f"unparsable observation: {item!r}")
                neg = bool(mt.group(1))
            observed[var] = not neg
    variables: List[str] = []
    for op, signs, parents, objects in clauses:
        for p in parents:
            if p not in variables:
                variables.append(p)
        for neg, v in objects:
            if v not in variables:
                variables.append(v)
    for v in list(priors) + list(observed):
        if v not in variables:
            variables.append(v)
    return CausalModel(
        priors=priors, observed=observed, variables=variables, clauses=clauses
    )


def build_equations(
    model: CausalModel,
    and_op: str = "and",
    or_op: str = "or",
    neg_mode: str = "keep",
) -> Dict[str, Equation]:
    """Compile raw clauses into equations under a semantics configuration.

    neg_mode:
      'keep'     - honor every 'not' (subject-side and object-side);
      'drop_or'  - drop subject-side negations inside OR-clauses only;
      'drop_all' - drop every negation (subject- and object-side).
    """
    equations: Dict[str, Equation] = {}
    for op, signs, parents, objects in model.clauses:
        for neg, var in objects:
            if var in equations:
                raise ParseError(f"duplicate mechanism for {var!r}")
            eop = {"id": "id", "and": and_op, "or": or_op}[op]
            esigns = list(signs)
            has_neg_input = not all(esigns)
            if neg_mode == "drop_or" and op == "or":
                esigns = [True] * len(esigns)
            elif neg_mode == "drop_all":
                esigns = [True] * len(esigns)
            elif neg_mode == "or_neg_and" and op == "or" and has_neg_input:
                eop = "and"  # negated cause inside an 'or' -> conjunctive
            out_neg = neg
            if neg_mode == "drop_all":
                out_neg = False
            equations[var] = Equation(eop, tuple(esigns), tuple(parents), out_neg)
    return equations


# ---------------------------------------------------------------------------
# Question parsing
# ---------------------------------------------------------------------------


@dataclass
class Query:
    qtype: str  # 'cf_single' | 'cf_joint' | 'cf_nested'
    target: str  # Y
    assume_vars: List[str]
    assume_vals: List[bool]


def parse_question(question: str) -> Query:
    q = question.strip()
    m = re.fullmatch(rf"Would ({_WORD}) occur if not ({_WORD}) instead of ({_WORD})\?", q)
    if m:
        if m.group(2) != m.group(3):
            raise ParseError(f"mismatched intervention in question: {q!r}")
        return Query("cf_single", m.group(1), [m.group(2)], [False])
    m = re.fullmatch(rf"Would ({_WORD}) occur if not ({_WORD}) and not ({_WORD})\?", q)
    if m:
        return Query("cf_joint", m.group(1), [m.group(2), m.group(3)], [False, False])
    m = re.fullmatch(
        rf"Assume not ({_WORD}), and based on this assumption, further suppose "
        rf"not ({_WORD})\. Would ({_WORD}) occur\?",
        q,
    )
    if m:
        return Query("cf_nested", m.group(3), [m.group(1), m.group(2)], [False, False])
    raise ParseError(f"unparsable question: {q!r}")


# ---------------------------------------------------------------------------
# Solver: abduction + action + prediction
# ---------------------------------------------------------------------------


@dataclass
class Semantics:
    and_op: str = "and"          # op used for 'A and B together cause C'
    or_op: str = "or"            # op used for 'A or B cause C'
    neg_mode: str = "keep"       # 'keep' | 'drop_or' | 'drop_all' | 'or_neg_and'
    factual: str = "relax"       # 'abduct' | 'relax' | 'prior' | 'true'
    obs_in_cf: bool = False      # pin observed values in the counterfactual?
    interv: str = "set_false"    # 'set_false' | 'flip'
    second: str = "apply"        # second premise: 'apply' | 'skip_conflict' | 'flip_factual'
    fallback: str = "no"


def _candidate_worlds(
    model: CausalModel,
    equations: Dict[str, Equation],
    observations: Dict[str, bool],
) -> List[Dict[str, bool]]:
    """ABDUCTION: (root assignment, fixed point) pairs consistent with obs."""
    roots = [v for v in model.variables if v not in equations]
    if len(roots) > 12:
        raise ParseError(f"too many roots: {roots}")
    m2 = CausalModel(
        equations=equations, priors=model.priors, observed={},
        variables=model.variables,
    )
    worlds: List[Dict[str, bool]] = []
    for combo in itertools.product([False, True], repeat=len(roots)):
        forced = dict(zip(roots, combo))
        for fp in m2.fixed_points(forced):
            if all(fp[k] == v for k, v in observations.items()):
                worlds.append(fp)
    return worlds


def _relaxed_observations(
    model: CausalModel,
    equations: Dict[str, Equation],
) -> List[Dict[str, bool]]:
    """All maximal observation subsets that admit a consistent world."""
    obs = model.observed
    if _candidate_worlds(model, equations, obs):
        return [obs]
    keys = list(obs)
    for size in range(len(keys) - 1, -1, -1):
        found = []
        for subset in itertools.combinations(keys, size):
            sub = {k: obs[k] for k in subset}
            if _candidate_worlds(model, equations, sub):
                found.append(sub)
        if found:
            return found
    return [{}]


def solve(given_info: str, question: str, sem: Semantics = Semantics()) -> str:
    """Answer a CounterBench question with 'yes'/'no'."""
    model = parse_context(given_info)
    query = parse_question(question)
    equations = build_equations(
        model, and_op=sem.and_op, or_op=sem.or_op, neg_mode=sem.neg_mode
    )
    m2 = CausalModel(
        equations=equations, priors=model.priors, observed={},
        variables=model.variables,
    )
    target = query.target

    # ---- factual worlds (abduction, with optional relaxation) -----------
    if sem.factual == "relax":
        worlds: List[Dict[str, bool]] = []
        for obs in _relaxed_observations(model, equations):
            worlds.extend(_candidate_worlds(model, equations, obs))
    else:
        worlds = _candidate_worlds(model, equations, model.observed)
    if not worlds:
        return sem.fallback
    if sem.factual == "prior":
        import math

        def score(w):
            s = 0.0
            for var in m2.roots:
                p = model.priors.get(var, 0.5)
                val = w[var]
                s += math.log(max(p if val else 1 - p, 1e-9))
            return s

        best = max(score(w) for w in worlds)
        worlds = [w for w in worlds if abs(score(w) - best) < 1e-9]
    elif sem.factual == "true":
        best = max(sum(1 for v in m2.roots if w[v]) for w in worlds)
        worlds = [w for w in worlds if sum(1 for v in m2.roots if w[v]) == best]

    # ---- counterfactual worlds (action + prediction) --------------------
    answers: List[bool] = []

    def cf_worlds(forced):
        fps = m2.fixed_points(forced)
        if not fps:  # pathological: keep forced values as-is
            fps = [dict(forced)]
        return fps

    for w in worlds:
        forced = {var: w[var] for var in m2.roots}
        if sem.obs_in_cf:
            for k, v in model.observed.items():
                forced[k] = v

        if query.qtype == "cf_single":
            x = query.assume_vars[0]
            forced[x] = False if sem.interv == "set_false" else (not w[x])
            for fp in cf_worlds(forced):
                answers.append(fp[target])
        elif query.qtype == "cf_joint":
            x1, x2 = query.assume_vars
            if sem.second == "flip_factual":
                f = dict(forced)
                f[x1] = False if sem.interv == "set_false" else (not w[x1])
                f[x2] = not w[x2]  # premise 'not x2' flips its factual value
                for fp in cf_worlds(f):
                    answers.append(fp[target])
                continue
            f1 = dict(forced)
            f1[x1] = False
            w1s = cf_worlds(f1)
            w1 = next(
                (c for c in w1s if all(c[v] == w[v] for v in m2.roots)), w1s[0]
            )
            if sem.second == "skip_conflict" and w1.get(x2) is True:
                answers.append(w1[target])
                continue
            f2 = dict(f1)
            f2[x2] = False
            for fp in cf_worlds(f2):
                answers.append(fp[target])
        else:  # cf_nested
            a, b = query.assume_vars
            if sem.second == "flip_factual":
                f = dict(forced)
                f[a] = False if sem.interv == "set_false" else (not w[a])
                f[b] = not w[b]  # premise 'not b' flips its factual value
                for fp in cf_worlds(f):
                    answers.append(fp[target])
                continue
            f1 = dict(forced)
            f1[a] = False if sem.interv == "set_false" else (not w[a])
            w1s = cf_worlds(f1)
            w1 = next(
                (c for c in w1s if all(c[v] == w[v] for v in m2.roots)), w1s[0]
            )
            if sem.second == "skip_conflict" and w1.get(b) is True:
                answers.append(w1[target])
                continue
            f2 = dict(f1)
            f2[b] = False if sem.interv == "set_false" else (not w1[b])
            for fp in cf_worlds(f2):
                answers.append(fp[target])

    if not answers:
        return sem.fallback
    yes = sum(1 for a in answers if a)
    if yes == len(answers):
        return "yes"
    if yes == 0:
        return "no"
    return "yes" if 2 * yes > len(answers) else sem.fallback


''')

In [4]:
# Part 3a — FOLIO: FOL tokenizer, parser and CNF conversion.
folio_cnf = _mod('folio_cnf', r'''"""FOLIO engine part 1: terms, FOL parser, CNF conversion."""
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Dict, FrozenSet, List, Optional, Set


@dataclass(frozen=True)
class Var:
    name: str


@dataclass(frozen=True)
class Const:
    name: str


@dataclass(frozen=True)
class Func:
    name: str
    args: tuple


@dataclass(frozen=True)
class Atom:
    pred: str
    args: tuple
    sign: bool = True


class ParseError(ValueError):
    pass


_TOKEN_RE = re.compile(
    r"(∀|∃|→|∨|∧|¬|⊕|↔|⟷|≠|[(),.]|=)|([^\W\d_][\w]*(?:-[\w]+)*)"
)


def tokenize(s: str) -> List[str]:
    s = s.replace("—", " ")
    toks: List[str] = []
    i = 0
    while i < len(s):
        if s[i].isspace():
            i += 1
            continue
        m = _TOKEN_RE.match(s, i)
        if not m:
            raise ParseError("bad char %r in %r" % (s[i], s))
        toks.append(m.group(0))
        i = m.end()
    toks = [t for t in toks if t != "."]
    # tolerate sloppy annotations: drop ')' that cannot match (left to right)
    out = []
    depth = 0
    for t in toks:
        if t == "(":
            depth += 1
            out.append(t)
        elif t == ")":
            if depth > 0:
                depth -= 1
                out.append(t)
            # else: drop unmatched ')'
        else:
            out.append(t)
    toks = out + [")"] * depth
    return toks


class _Parser:
    def __init__(self, toks: List[str]):
        self.toks = toks
        self.pos = 0
        self.bound: List[str] = []

    def peek(self) -> Optional[str]:
        return self.toks[self.pos] if self.pos < len(self.toks) else None

    def next(self) -> str:
        t = self.peek()
        if t is None:
            raise ParseError("unexpected end of formula")
        self.pos += 1
        return t

    def parse(self):
        f = self.parse_iff()
        if self.peek() is not None:
            raise ParseError("trailing tokens: %r" % (self.toks[self.pos:],))
        return f

    def parse_iff(self):
        left = self.parse_xor()
        while self.peek() in ("↔", "⟷"):
            self.next()
            left = ("iff", left, self.parse_xor())
        return left

    def parse_xor(self):
        left = self.parse_imp()
        while self.peek() == "⊕":
            self.next()
            left = ("xor", left, self.parse_imp())
        return left

    def parse_imp(self):
        left = self.parse_or()
        if self.peek() == "→":
            self.next()
            return ("imp", left, self.parse_imp())
        return left

    def parse_or(self):
        left = self.parse_and()
        while self.peek() == "∨":
            self.next()
            left = ("or", left, self.parse_and())
        return left

    def parse_and(self):
        left = self.parse_unary()
        while self.peek() == "∧":
            self.next()
            left = ("and", left, self.parse_unary())
        return left

    def parse_unary(self):
        t = self.peek()
        if t == "¬":
            self.next()
            return ("not", self.parse_unary())
        if t in ("∀", "∃"):
            self.next()
            v = self.next()
            if not re.fullmatch(r"[^\W\d_][\w]*(?:-[\w]+)*", v or ""):
                raise ParseError("bad quantifier var %r" % (v,))
            self.bound.append(v)
            body = self.parse_unary()
            self.bound.pop()
            return ("forall" if t == "∀" else "exists", v, body)
        return self.parse_primary()

    def parse_primary(self):
        t = self.peek()
        if t == "(":
            self.next()
            f = self.parse_iff()
            if self.next() != ")":
                raise ParseError("missing ')'")
            return f
        if t is not None and re.fullmatch(r"[^\W\d_][\w]*(?:-[\w]+)*", t):
            name = self.next()
            if self.peek() == "(":
                self.next()
                args = []
                if self.peek() != ")":
                    args.append(self.parse_term())
                    while self.peek() == ",":
                        self.next()
                        args.append(self.parse_term())
                if self.next() != ")":
                    raise ParseError("missing ')' in atom")
                return Atom(name, tuple(args), True)
            if self.peek() == "=":
                self.next()
                return ("eq", self._to_term(name), self.parse_term())
            if self.peek() == "≠":
                self.next()
                return ("neq", self._to_term(name), self.parse_term())
            # bare predicate/constant
            return Atom(name, (), True)
        raise ParseError("unexpected token %r" % (t,))

    def parse_term(self):
        t = self.peek()
        if t is not None and re.fullmatch(r"[^\W\d_][\w]*(?:-[\w]+)*", t):
            name = self.next()
            if self.peek() == "(":
                self.next()
                args = []
                if self.peek() != ")":
                    args.append(self.parse_term())
                    while self.peek() == ",":
                        self.next()
                        args.append(self.parse_term())
                if self.next() != ")":
                    raise ParseError("missing ')' in term")
                return Func(name, tuple(args))
            return self._to_term(name)
        raise ParseError("bad term token %r" % (t,))

    def _to_term(self, name: str):
        return Var(name) if name in self.bound else Const(name)


def parse_formula(s: str):
    return _Parser(tokenize(s)).parse()


_counter = [0]


def _elim(f):
    if isinstance(f, Atom):
        return f
    tag = f[0]
    if tag == "not":
        return ("not", _elim(f[1]))
    if tag in ("and", "or"):
        return (tag, _elim(f[1]), _elim(f[2]))
    if tag == "imp":
        return ("or", ("not", _elim(f[1])), _elim(f[2]))
    if tag == "iff":
        a, b = _elim(f[1]), _elim(f[2])
        return ("and", ("or", ("not", a), b), ("or", ("not", b), a))
    if tag == "xor":
        a, b = _elim(f[1]), _elim(f[2])
        return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    if tag == "eq":
        return Atom("Eq", (f[1], f[2]), True)
    if tag == "neq":
        return Atom("Eq", (f[1], f[2]), False)
    if tag in ("forall", "exists"):
        return (tag, f[1], _elim(f[2]))
    raise ParseError("bad node %r" % (f,))


def _nnf(f, neg: bool = False):
    if isinstance(f, Atom):
        return Atom(f.pred, f.args, (not f.sign) if neg else f.sign)
    tag = f[0]
    if tag == "not":
        return _nnf(f[1], not neg)
    if neg:
        if tag == "and":
            return ("or", _nnf(f[1], True), _nnf(f[2], True))
        if tag == "or":
            return ("and", _nnf(f[1], True), _nnf(f[2], True))
        if tag in ("forall", "exists"):
            flip = "exists" if tag == "forall" else "forall"
            return (flip, f[1], _nnf(f[2], True))
        raise ParseError("bad node %r" % (f,))
    if tag in ("and", "or"):
        return (tag, _nnf(f[1]), _nnf(f[2]))
    if tag in ("forall", "exists"):
        return (tag, f[1], _nnf(f[2]))
    raise ParseError("bad node %r" % (f,))


def _subst(f, env: Dict[str, object]):
    if isinstance(f, Atom):
        return Atom(f.pred, tuple(_subst_t(a, env) for a in f.args), f.sign)
    tag = f[0]
    if tag == "not":
        return ("not", _subst(f[1], env))
    if tag in ("and", "or"):
        return (tag, _subst(f[1], env), _subst(f[2], env))
    if tag in ("forall", "exists"):
        env2 = {k: v for k, v in env.items() if k != f[1]}
        return (tag, f[1], _subst(f[2], env2))
    raise ParseError("bad node %r" % (f,))


def _subst_t(t, env: Dict[str, object]):
    if isinstance(t, Var):
        return env.get(t.name, t)
    if isinstance(t, Func):
        return Func(t.name, tuple(_subst_t(a, env) for a in t.args))
    return t


def _rename_for_suffix(f, old: str, suffix: str):
    if isinstance(f, Atom):
        def rn(t):
            if isinstance(t, Var) and t.name == old:
                return Var("%s%s" % (old, suffix))
            if isinstance(t, Func):
                return Func(t.name, tuple(rn(a) for a in t.args))
            return t
        return Atom(f.pred, tuple(rn(a) for a in f.args), f.sign)
    tag = f[0]
    if tag == "not":
        return ("not", _rename_for_suffix(f[1], old, suffix))
    if tag in ("and", "or"):
        return (tag, _rename_for_suffix(f[1], old, suffix),
                _rename_for_suffix(f[2], old, suffix))
    if tag in ("forall", "exists"):
        if f[1] == old:
            return f
        return (tag, f[1], _rename_for_suffix(f[2], old, suffix))
    raise ParseError("bad node %r" % (f,))


def _standardize(f, suffix: str):
    if isinstance(f, Atom):
        return f
    tag = f[0]
    if tag == "not":
        return ("not", _standardize(f[1], suffix))
    if tag in ("and", "or"):
        return (tag, _standardize(f[1], suffix), _standardize(f[2], suffix))
    if tag in ("forall", "exists"):
        return (tag, "%s%s" % (f[1], suffix),
                _rename_for_suffix(_standardize(f[2], suffix), f[1], suffix))
    raise ParseError("bad node %r" % (f,))


def _skolemize(f, universals: List[str]):
    if isinstance(f, Atom):
        return f
    tag = f[0]
    if tag == "forall":
        return ("forall", f[1], _skolemize(f[2], universals + [f[1]]))
    if tag == "exists":
        v = f[1]
        _counter[0] += 1
        if universals:
            sk = Func("sk%d" % _counter[0], tuple(Var(u) for u in universals))
        else:
            sk = Const("sk%d" % _counter[0])
        return _skolemize(_subst(f[2], {v: sk}), universals)
    if tag == "not":
        return ("not", _skolemize(f[1], universals))
    if tag in ("and", "or"):
        return (tag, _skolemize(f[1], universals),
                _skolemize(f[2], universals))
    raise ParseError("bad node %r" % (f,))


def _drop_foralls(f):
    if isinstance(f, Atom):
        return f
    tag = f[0]
    if tag == "forall":
        return _drop_foralls(f[2])
    if tag == "not":
        return ("not", _drop_foralls(f[1]))
    if tag in ("and", "or"):
        return (tag, _drop_foralls(f[1]), _drop_foralls(f[2]))
    raise ParseError("bad node %r" % (f,))


def _distribute(f) -> List[FrozenSet[Atom]]:
    if isinstance(f, Atom):
        return [frozenset([f])]
    tag = f[0]
    if tag == "not":
        if not isinstance(f[1], Atom):
            raise ParseError("NNF violated")
        return [frozenset([f[1]])]
    if tag == "and":
        return _distribute(f[1]) + _distribute(f[2])
    if tag == "or":
        out = []
        for c1 in _distribute(f[1]):
            for c2 in _distribute(f[2]):
                out.append(c1 | c2)
        return out
    raise ParseError("bad node %r" % (f,))


def _free_vars(f) -> Set[str]:
    if isinstance(f, Atom):
        return {a.name for a in f.args if isinstance(a, Var)}
    tag = f[0]
    if tag == "not":
        return _free_vars(f[1])
    if tag in ("and", "or"):
        return _free_vars(f[1]) | _free_vars(f[2])
    if tag in ("forall", "exists"):
        return _free_vars(f[2]) - {f[1]}
    return set()


def to_cnf_ast(f, suffix: str) -> List[FrozenSet[Atom]]:
    free = _free_vars(f)
    if free:
        env = {}
        for v in free:
            _counter[0] += 1
            env[v] = Const("free%d" % _counter[0])
        f = _subst(f, env)
    f = _elim(f)
    f = _nnf(f)
    f = _standardize(f, suffix)
    f = _skolemize(f, [])
    f = _drop_foralls(f)
    return _distribute(f)


def to_cnf(formula_str: str, suffix: str) -> List[FrozenSet[Atom]]:
    return to_cnf_ast(parse_formula(formula_str), suffix)

''')

In [5]:
# Part 3b — FOLIO: NL→FOL template parser (fallback when no gold FOL is given).
folio_nl = _mod('folio_nl', r'''"""NL -> FOL fallback parser for FOLIO-style English (no gold annotations).

Strategy (all deterministic rules, no ML):

  1. Per story, detect individual constants (proper names) shared by the
     premises and the conclusion.
  2. Parse every sentence into a logical skeleton with regex/cue templates:
     quantifiers (all/no/some/everyone/there-is/if-then), connectives
     (and/or/either-or/neither-nor/not) and property phrases.
  3. Map each property phrase to a predicate symbol by normalizing
     (lowercase, crude verb stemming, stopword removal) so paraphrases
     across sentences share one symbol; direct/prepositional objects
     become extra constant arguments.
  4. Emit FOL-dialect strings consumed by folio_cnf/folio_prover.

Coverage-first: anything unparsed becomes an opaque uninterpreted
predicate applied to the sentence subject (sound: distinct sentences get
distinct symbols, so no spurious proofs), and unparseable structure
abstains downstream (predict False).
"""
from __future__ import annotations

import re
from typing import Dict, List, Optional, Set, Tuple

STOP = {
    "a", "an", "the", "very", "really", "quite", "always", "usually",
    "often", "sometimes", "never", "just", "also", "both", "each",
}

GENERIC_CLASSES = {
    "people", "person", "humans", "human", "man", "woman", "men", "women",
    "thing", "things", "one", "ones", "individual", "individuals",
    "everyone", "everybody", "everyone", "someone", "somebody", "anyone",
    "anybody", "noone", "nobody", "nothing", "something", "anything",
    "everything",
}

PRONOUNS = {"she": None, "he": None, "her": None, "him": None,
            "they": None, "them": None, "it": None}

QUANT_WORDS = {"all", "every", "everyone", "everybody", "everything", "some",
               "someone", "somebody", "something", "no", "nobody", "noone",
               "none", "any", "there", "if", "either", "neither", "one",
               "people", "humans", "everyday"}

BE = {"is", "are", "was", "were", "be", "been", "being", "am"}


def stem(word: str) -> str:
    w = word.lower()
    # normalize year/number variants: yr1946 / year1946 / 1946 -> num1946
    m = re.fullmatch(r"(yr|year)?(\d+)", w)
    if m:
        return "num" + m.group(2)
    if w in BE:
        return "be"
    for suf in ("ies", "es", "ed", "ing", "ly"):
        if len(w) > len(suf) + 2 and w.endswith(suf):
            if suf == "ies":
                return w[:-3] + "y"
            if suf == "es" and w[-3] in "sxz" or w.endswith(("shes", "ches")):
                return w[:-2]
            if suf == "es" and w.endswith("oes"):
                return w[:-2]
            return w[:-len(suf)] if suf != "es" else w[:-1]
    if len(w) > 3 and w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w


def camel(words: List[str]) -> str:
    words = [w for w in words if w]
    if not words:
        return "Thing"
    return "".join(w[0].upper() + w[1:] for w in words)


def split_top(s: str, seps: List[str]) -> List[str]:
    """Split on the first matching separator (case-insensitive)."""
    low = s.lower()
    for sep in seps:
        i = low.find(sep.lower())
        if i >= 0:
            return [s[:i], s[i + len(sep):]]
    return [s]


def split_and_or(s: str, word: str) -> List[str]:
    return [p.strip() for p in re.split(r",\s*|\s+%s\s+" % word, s) if p.strip()]


class StoryContext:
    """Per-story constants + predicate symbol table (paraphrase alignment)."""

    def __init__(self, sentences: List[str]):
        self.names = self._find_names(sentences)
        self.pred_cache: Dict[str, str] = {}

    @staticmethod
    def _find_names(sentences: List[str]) -> List[str]:
        cands = {}
        for s in sentences:
            for m in re.finditer(r"[A-Z][a-z]+", s):
                w = m.group(0)
                if w in ("All", "Some", "Every", "Everyone", "No", "None",
                         "There", "Either", "Neither", "People", "Humans",
                         "One", "Any", "If", "The", "A", "An", "It", "In",
                         "Everyday", "Everything", "Everybody", "Someone",
                         "Something", "Nobody", "Nothing", "Customers"):
                    continue
                if m.start() == 0 and s.split()[0] == w:
                    # sentence-initial capitalized common noun? keep if seen
                    # elsewhere non-initially
                    pass
                cands[w] = cands.get(w, 0) + 1
        # keep names seen >=1 time but not sentence-initial-only common nouns
        return sorted(cands)

    def resolve_pronoun(self, mention: str) -> str:
        """Resolve a pronoun to the salient constant (single-name stories)."""
        m = mention.strip().lower()
        if m in ("she", "he", "her", "him", "they", "them", "it"):
            if len(self.names) == 1:
                n = self.names[0]
                return n[0].lower() + n[1:]
        return mention

    def const(self, mention: str) -> str:
        m = mention.strip()
        for n in self.names:
            if m.lower() == n.lower() or m.lower().endswith(" " + n.lower()):
                return n[0].lower() + n[1:]
        # generic kind constant
        return m.lower().replace(" ", "_")[:24]

    def is_name(self, mention: str) -> bool:
        m = mention.strip()
        return any(m.lower() == n.lower() or
                   m.lower().endswith(" " + n.lower()) for n in self.names)


# ---------------------------------------------------------------------------
# property phrase -> (pred_symbol, extra_args, polarity)
# ---------------------------------------------------------------------------

NEG_WORDS = {"not", "n't", "never", "no"}


def parse_property(phrase: str, ctx: StoryContext, subj: str):
    """Parse a property phrase into (fol_atom_string, )."""
    p = phrase.strip().rstrip(".").strip()
    # normalize possessives: "Mary's school" -> "Mary school"
    p = re.sub(r"'s\b", "", p)
    p = p.replace("'", "")
    # strip locative domain restrictors ('people at Mary school work ...'
    # -> 'people work ...'), but never the whole phrase
    q = re.sub(r"(?i)\bat\s+[A-Z][\w\-]*(?:\s+"
               r"(?:school|university|college|company|club|team|hospital))?",
               "", p).strip()
    q = re.sub(r"\s+", " ", q).strip()
    if q and re.search(r"[a-zA-Z]", q):
        p = q
    # resolve pronominal subject baked into phrase ('she is ...')
    m = re.match(r"(?i)^(she|he|they|it|her|him|them)\s+(is|are|was|were)\s+(.*)$", p)
    if m:
        subj = ctx.const(ctx.resolve_pronoun(m.group(1)))
        p = m.group(3).strip()
    # leading copula left over after subject extraction
    p = re.sub(r"(?i)^(is|are|was|were)\s+", "", p).strip()
    neg = False
    # leading negation
    m = re.match(r"(?i)^(not|never)\s+(.*)$", p)
    if m:
        neg = True
        p = m.group(2).strip()
    # auxiliaries: n't-forms negate, plain forms don't
    m = re.match(r"(?i)^(doesn't|don't|doesnt|dont|isn't|arent|isnt|aren't|"
                 r"wasn't|werent|wasnt|weren't|can't|cant|cannot|won't|wont)\s+(.*)$", p)
    if m:
        neg = not neg
        p = m.group(2).strip()
    m = re.match(r"(?i)^(does|do|is|are|was|were|did|has|have|had|will|"
                 r"would|can|could|should|must)\s+(.*)$", p)
    if m:
        p = m.group(2).strip()
    # trailing 'or both' etc.
    p = re.sub(r"(?i)\s*,?\s*or both\s*$", "", p).strip()
    words = p.split()
    # strip leading articles / auxiliaries leftovers
    while words and words[0].lower() in ("a", "an", "the"):
        words = words[1:]
    if not words:
        return None
    # negation inside ('not X', "n't")
    if words[0].lower() in ("not", "n't") or words[0].endswith("n't"):
        neg = not neg
        words = words[1:]
        if words and words[0].lower() in ("a", "an", "the"):
            words = words[1:]
    if not words:
        return None
    # relative clause inside NP: 'a student who is unaware ...'
    for rel in (" who ", " that ", " which "):
        if rel in (" " + " ".join(words) + " "):
            break
    else:
        rel = None
    rel_cond = None
    if rel:
        joined = " ".join(words)
        idx = joined.find(rel)
        head_np, rel_txt = joined[:idx].strip(), joined[idx + len(rel):].strip()
        # class atom from head noun (last word)
        head_words = [w for w in head_np.split()
                      if w.lower() not in ("a", "an", "the")]
        if head_words:
            cls = stem(head_words[-1])
            if cls not in GENERIC_CLASSES and re.fullmatch(r"[a-z]+", cls):
                rel_cond = ("cls", camel([cls]))
        rel_parsed = parse_side(rel_txt, ctx, subj)
        if rel_parsed is None:
            return None
        if rel_cond is None:
            return ("¬(%s)" % rel_parsed) if neg else rel_parsed
        cls_atom = "%s(%s)" % (rel_cond[1], subj)
        combo = "%s ∧ (%s)" % (cls_atom, rel_parsed)
        return ("¬(%s)" % combo) if neg else combo
    # prepositional tail -> extra constant arg: 'dependent on caffeine'
    extra = None
    tail_words: List[str] = []
    for i in range(len(words) - 2, -1, -1):
        if words[i].lower() in ("on", "in", "to", "of", "with", "for",
                                "from", "at", "as") and i + 1 < len(words):
            tail = [w for w in words[i + 1:] if w.lower() not in STOP]
            if tail and len(tail) <= 3:
                # keep the tail as the argument only if a head remains;
                # otherwise ('at Mary school') it IS the predicate
                if [w for w in words[:i] if w.lower() not in STOP]:
                    extra = "_".join(t.lower() for t in tail)
                    words = words[:i]
                    break
                tail_words = tail
                words = words[:i] + tail
                break
    # direct object tail: 'drink coffee' / 'love music'
    if extra is None and len(words) >= 2:
        last = words[-1]
        if (last[0].islower() and last.lower() not in STOP
                and stem(last) != stem(words[-2])):
            # bare-noun object heuristic: single trailing lowercase noun
            # that is not a verb form of the head
            if re.fullmatch(r"[a-z]+", last):
                extra = last.lower()
                words = words[:-1]
    content = [stem(w) for w in words if w.lower() not in STOP]
    content = [w for w in content if re.fullmatch(r"[a-z][a-z0-9]*", w)]
    if not content:
        return None
    sym = camel(content)
    key = sym
    if key not in ctx.pred_cache:
        ctx.pred_cache[key] = sym
    args = [subj] + ([extra] if extra else [])
    atom = "%s(%s)" % (ctx.pred_cache[key], ", ".join(args))
    if neg:
        atom = "¬" + atom
    return atom


def _strip_subj(part: str, subj: str) -> str:
    """Remove a leading subject mention (pronoun, generic NP or the subject).

    'they don't want ...' (subj=x) -> "don't want ...".
    'she is not a student' (subj=rina) -> 'is not a student'.
    'someone is white' (subj=x) -> 'is white'.
    """
    low = subj.lower()
    names = {"they", "she", "he", "it", "we", "you", "i", "them", "her",
             "him", "his", "someone", "somebody", "something", "everyone",
             "everybody", "everything", "anyone", "anything", "noone",
             "nobody", "nothing", "people", "humans", "everyday"}
    if low != "x":
        names.add(low)
        names.add("the " + low)
    m = re.match(r"(?i)^(%s)\s+(.*)$" % "|".join(sorted(names, key=len,
                                                       reverse=True)),
                 part.strip())
    if m:
        return m.group(2).strip()
    # leading clause connective left over from or/and splitting
    m = re.match(r"(?i)^(or|and|nor)\s+(.*)$", part.strip())
    if m:
        return m.group(2).strip()
    return part.strip()


def parse_conjunction(txt: str, ctx: StoryContext, subj: str,
                      op: str = "∧") -> Optional[str]:
    parts = split_and_or(txt, "and")
    atoms = []
    for p in parts:
        p = _strip_subj(_fix_inversion(p), subj)
        if not p:
            continue
        # nested implication inside a conjunction
        if p.lower().startswith("if "):
            a = parse_implication(p, ctx)
            if a is None:
                return None
            atoms.append("(%s)" % a)
            continue
        # elliptical continuation: bare (negated) adjective -> copula
        if re.fullmatch(r"(?i)(not\s+)?[a-z]+", p):
            p = "is " + p
        a = parse_property(p, ctx, subj)
        if a is None:
            return None
        atoms.append(a)
    if not atoms:
        return None
    return (" %s " % op).join("(%s)" % a if op in a else a for a in atoms)


def _fix_inversion(part: str) -> str:
    """Repair inverted 'is she aware ...' (typo for 'she is aware ...').

    Only fires for pronoun subjects or capitalized names, so plain
    copular phrases ('is unaware ...') are never rewritten.
    """
    m = re.match(r"(?i)^(is|are|was|were)\s+"
                 r"(she|he|they|it|we|you|i|her|him|them)\s+(.*)$",
                 part.strip())
    if m:
        return "%s is %s" % (m.group(2), m.group(3))
    m = re.match(r"^(is|are|was|were)\s+([A-Z][\w\-]*)\s+(.*)$",
                 part.strip())
    if m:
        return "%s is %s" % (m.group(2), m.group(3))
    return part


def parse_alt(txt: str, ctx: StoryContext, subj: str, xor: bool) -> Optional[str]:
    t = re.sub(r"(?i)\s*,?\s*or both\s*$", "", txt.strip()).strip()
    t = _clean_conn(t)
    if not re.search(r"\bor\b", t.lower()):
        # no real disjunction left (e.g. after cleaning) -> conjunction
        return parse_conjunction(t, ctx, subj)
    parts = split_and_or(t, "or")
    atoms = []
    for p in parts:
        p = _clean_conn(_strip_subj(_fix_inversion(p), subj))
        if not p or p.lower() == "both":
            continue
        # disjuncts may themselves be conjunctions or neither-nor
        a = parse_side(p, ctx, subj)
        if a is None:
            return None
        atoms.append(a)
    if not atoms:
        return None
    op = "⊕" if xor else "∨"
    return (" %s " % op).join(atoms)


def _subject_span(s: str, ctx: StoryContext) -> Tuple[str, str, bool]:
    """Split 'SUBJ REST' -> (subject_fol, rest, is_variable).

    Returns the FOL subject term, the remaining predicate text, and whether
    the subject is a quantified variable (generic) or a constant.
    """
    s = s.strip()
    # pronouns resolve to the single salient name when there is one
    m = re.match(r"(?i)^(she|he|they|it|her|him|them)\b\s*(.*)$", s)
    if m and len(ctx.names) == 1:
        n = ctx.names[0]
        return n[0].lower() + n[1:], m.group(2).strip(), False
    # quantified/generic subjects
    m = re.match(r"(?i)^(everyone|everybody|everything|people|humans|"
                 r"someone|somebody|something|no\s+one|nobody|nothing|"
                 r"anyone|anything|they|it|we|you)\b\s*(.*)$", s)
    if m:
        return "x", m.group(2).strip(), True
    # 'there is/are ...'
    m = re.match(r"(?i)^there\s+(is|are)\s+(.*)$", s)
    if m:
        return "x", m.group(2).strip(), True
    # proper-name subject: leading capitalized words
    m = re.match(r"^((?:[A-Z][\w\-]*)(?:\s+[A-Z][\w\-]*)*)\s+(.*)$", s)
    if m and ctx.is_name(m.group(1)):
        return ctx.const(m.group(1)), m.group(2).strip(), False
    return "x", s, True


def _sym_words(sym: str) -> Set[str]:
    parts = re.findall(r"[A-Z]?[a-z]+|num[0-9]+|[0-9]+", sym)
    return {p.lower() for p in parts}


def align_conclusion(prems: List[str], concl: str) -> str:
    """Remap conclusion predicate symbols to premise symbols (paraphrases).

    For each predicate symbol in the conclusion that never occurs in the
    premises with the same arity, substitute the premise symbol with the
    same arity and the highest content-word overlap, provided the overlap
    (Jaccard over camel-case word parts) is >= 0.5.  This resolves
    paraphrase drift such as 'is dependent on' vs 'depends on' while
    leaving unrelated symbols untouched.
    """
    prem_syms = {}
    for atom_m in re.finditer(r"([A-Z][\w]*)\(([^()]*)\)", " ".join(prems)):
        sym, args = atom_m.group(1), atom_m.group(2)
        arity = 0 if not args.strip() else len(args.split(","))
        prem_syms.setdefault((sym, arity), sym)
    by_arity: Dict[int, List[str]] = {}
    for (sym, arity) in prem_syms:
        if sym.startswith("Opaque"):
            continue
        by_arity.setdefault(arity, []).append(sym)

    def repl(m):
        sym, args = m.group(1), m.group(2)
        arity = 0 if not args.strip() else len(args.split(","))
        if (sym, arity) in prem_syms or sym.startswith("Opaque"):
            return m.group(0)
        cands = by_arity.get(arity, [])
        best, best_j = None, 0.0
        sw = _sym_words(sym)
        for c in cands:
            cw = _sym_words(c)
            union = sw | cw
            j = len(sw & cw) / len(union) if union else 0.0
            if j > best_j:
                best, best_j = c, j
        if best is not None and best_j >= 0.5:
            return "%s(%s)" % (best, args)
        return m.group(0)

    return re.sub(r"([A-Z][\w]*)\(([^()]*)\)", repl, concl)


def parse_sentence(s: str, ctx: StoryContext) -> Optional[str]:
    """Parse one NL sentence into a FOL-dialect string (or None)."""
    s = s.strip().rstrip(".").strip()
    if not s:
        return None
    # normalize known dataset typos/truncations
    s = re.sub(r"(?i)\beith\b", "either", s)
    s = re.sub(r"\s+", " ", s).strip()
    low = s.lower()

    # -- if P then Q / if P, Q --
    if low.startswith("if "):
        r = parse_implication(s, ctx)
        return r

    # -- all/every X ... --
    m = re.match(r"(?i)^(all|every)\s+(.*)$", s)
    if m:
        rest = m.group(2).strip()
        # 'all X who/that P are Q' / 'all X are Q'
        parts = split_top(rest, [" are ", " is "])
        if len(parts) == 2:
            subj_txt, head_txt = parts
            cls, cond = _split_relative(subj_txt)
            atoms = []
            if cls and stem(cls.split()[-1]) not in GENERIC_CLASSES:
                c = parse_property(cls, ctx, "x")
                if c is None:
                    return None
                atoms.append(c)
            if cond:
                c = parse_side(cond, ctx, "x")
                if c is None:
                    return None
                atoms.append("(%s)" % c)
            h = parse_side(head_txt, ctx, "x")
            if h is None:
                return None
            if not atoms:
                return "∀x (%s)" % h
            body = " ∧ ".join(atoms)
            return "∀x ((%s) → (%s))" % (body, h)
        return None

    # -- no X ... / no one ... --
    m = re.match(r"(?i)^no\s+(.*)$", s)
    if m:
        rest = m.group(1).strip()
        parts = split_top(rest, [" are ", " is "])
        if len(parts) == 2:
            subj_txt, head_txt = parts
            cls, cond = _split_relative(subj_txt)
            atoms = []
            if cls and stem(cls.split()[-1]) not in GENERIC_CLASSES:
                c = parse_property(cls, ctx, "x")
                if c is None:
                    return None
                atoms.append(c)
            if cond:
                c = parse_side(cond, ctx, "x")
                if c is None:
                    return None
                atoms.append("(%s)" % c)
            h = parse_side(head_txt, ctx, "x")
            if h is None:
                return None
            if not atoms:
                # 'no one is Q' / 'no X is Q' with vacuous class
                return "∀x (¬(%s))" % h
            return "∀x ((%s) → ¬(%s))" % (" ∧ ".join(atoms), h)
        body = parse_side(rest, ctx, "x")
        if body is None:
            return None
        return "∀x (¬(%s))" % body

    # -- none ... --
    if low.startswith("none "):
        body = parse_side(s[5:].strip(), ctx, "x")
        if body is None:
            return None
        return "∀x (¬(%s))" % body

    # -- some X ... / there is/are ... --
    m = re.match(r"(?i)^(some\s+.*|there\s+(is|are)\s+.*)$", s)
    if m:
        rest = m.group(1)
        rest = re.sub(r"(?i)^there\s+(is|are)\s+", "", rest).strip()
        rest = re.sub(r"(?i)^some\s+", "", rest).strip()
        body = parse_side(rest, ctx, "x")
        if body is None:
            return None
        return "∃x (%s)" % body

    # -- everyone/everybody ... --
    m = re.match(r"(?i)^(everyone|everybody|everything)\s+(.*)$", s)
    if m:
        body = parse_side(m.group(2), ctx, "x")
        if body is None:
            return None
        return "∀x (%s)" % body

    # -- plain assertion (constant subject or generic) --
    subj, rest, is_var = _subject_span(s, ctx)
    body = parse_side(rest, ctx, subj)
    if body is None:
        return None
    if is_var:
        # guess quantifier: bare generic assertions default to universal
        return "∀x (%s)" % body
    return body


def _split_relative(subj_txt: str) -> Tuple[Optional[str], Optional[str]]:
    """'men who love music' -> ('men', 'love music')."""
    parts = split_top(subj_txt, [" who ", " that ", " which "])
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    return subj_txt.strip(), None


def _clean_conn(txt: str) -> str:
    """Strip dangling clause connectives left by comma-splitting."""
    t = txt.strip()
    t = re.sub(r"(?i)^[\s,]*(or|and|nor|but)\b[\s,]*", "", t).strip()
    t = re.sub(r"(?i)[\s,]*\b(or|and|nor|but)[\s,]*$", "", t).strip()
    return t


_SIDE_DEPTH = [0]
_SIDE_MAX = 24


def parse_implication(txt: str, ctx: StoryContext) -> Optional[str]:
    """Parse 'if P then Q' / 'if P, Q' (also nested inside sides)."""
    rest = re.sub(r"(?i)^if\s+", "", txt.strip()).strip()
    parts = split_top(rest, [" then ", ", then "])
    if len(parts) == 2:
        cond, conseq = parts
    else:
        parts = split_top(rest, [", "])
        if len(parts) != 2:
            return None
        cond, conseq = parts
    subj_c, rest_c, _ = _subject_span(cond, ctx)
    subj_q, rest_q, _ = _subject_span(conseq, ctx)
    subj = subj_c if subj_c != "x" else subj_q
    a = parse_side(cond if subj_c != "x" else rest_c, ctx, subj)
    b = parse_side(conseq if subj_q != "x" else rest_q, ctx, subj)
    if a is None or b is None:
        return None
    if subj == "x":
        return "∀x ((%s) → (%s))" % (a, b)
    return "((%s) → (%s))" % (a, b)


def parse_side(txt: str, ctx: StoryContext, subj: str) -> Optional[str]:
    """Parse a condition/consequent side with and/or/either/neither/not."""
    if _SIDE_DEPTH[0] >= _SIDE_MAX:
        return None
    _SIDE_DEPTH[0] += 1
    try:
        return _parse_side_inner(txt, ctx, subj)
    finally:
        _SIDE_DEPTH[0] -= 1


def _parse_side_inner(txt: str, ctx: StoryContext, subj: str) -> Optional[str]:
    t = txt.strip()
    # strip a leading copula left over after subject extraction
    t = re.sub(r"(?i)^(is|are|was|were)\s+", "", t).strip()
    if not t:
        return None
    low = t.lower()
    # nested implication ('..., or, if she is, then ...')
    if low.startswith("if ") and (", " in t or " then " in low):
        return parse_implication(t, ctx)
    # neither-nor
    if low.startswith("neither "):
        rest = re.sub(r"(?i)^neither\s+", "", t).strip()
        parts = split_and_or(rest, "nor")
        atoms = []
        for p in parts:
            p = _clean_conn(_strip_subj(_fix_inversion(p), subj))
            if not p:
                continue
            a = parse_property(p, ctx, subj)
            if a is None:
                return None
            atoms.append("¬(%s)" % a if not a.startswith("¬") else a[1:])
        return " ∧ ".join(atoms) if atoms else None
    # either-or
    if low.startswith("either "):
        rest = re.sub(r"(?i)^either\s+", "", t).strip()
        return parse_alt(rest, ctx, subj, xor=True)
    # plain or (check 'or both' -> inclusive)
    if re.search(r"\bor\b", low):
        c = _clean_conn(t)
        if not re.search(r"\bor\b", c.lower()):
            return parse_conjunction(c, ctx, subj)
        return parse_alt(c, ctx, subj, xor=False)
    return parse_conjunction(t, ctx, subj)


def parse_story(premises: List[str], conclusion: str
                ) -> Tuple[List[str], str, dict]:
    """Parse a full story. Returns (premise_fols, conclusion_fol, info)."""
    ctx = StoryContext(premises + [conclusion])
    prems = []
    failed = 0
    for p in premises:
        f = parse_sentence(p, ctx)
        if f is None:
            failed += 1
            # opaque fallback: uninterpreted predicate on subject
            subj, _, is_var = _subject_span(p, ctx)
            tag = "Opaque%d" % failed
            prems.append("%s(%s)" % (tag, subj))
        else:
            prems.append(f)
    concl = parse_sentence(conclusion, ctx)
    if concl is None:
        subj, _, _ = _subject_span(conclusion, ctx)
        concl = "OpaqueConcl(%s)" % subj
        failed += 1
    else:
        concl = align_conclusion(prems, concl)
    return prems, concl, {"failed": failed,
                         "predicates": sorted(ctx.pred_cache.values())}


''')

In [6]:
# Part 3c — FOLIO: resolution refutation prover.
folio_prover = _mod('folio_prover', r'''"""FOLIO engine part 2: unification, resolution refutation, top-level answer."""
from __future__ import annotations

import time
from typing import Dict, FrozenSet, List, Optional, Set, Tuple

from folio_cnf import Atom, Const, Func, ParseError, Var, parse_formula, to_cnf, to_cnf_ast


def _occurs(v: str, t) -> bool:
    if isinstance(t, Var):
        return t.name == v
    if isinstance(t, Func):
        return any(_occurs(v, a) for a in t.args)
    return False


def _apply(t, env: Dict[str, object]):
    if isinstance(t, Var):
        v = env.get(t.name)
        if v is None:
            return t
        return _apply(v, env)
    if isinstance(t, Func):
        return Func(t.name, tuple(_apply(a, env) for a in t.args))
    return t


def unify_term(a, b, env: Dict[str, object]) -> Optional[Dict[str, object]]:
    a = _apply(a, env)
    b = _apply(b, env)
    if isinstance(a, Var):
        if isinstance(b, Var) and a.name == b.name:
            return env
        if _occurs(a.name, b):
            return None
        env[a.name] = b
        return env
    if isinstance(b, Var):
        return unify_term(b, a, env)
    if isinstance(a, Const) and isinstance(b, Const):
        return env if a.name == b.name else None
    if isinstance(a, Func) and isinstance(b, Func):
        if a.name != b.name or len(a.args) != len(b.args):
            return None
        for x, y in zip(a.args, b.args):
            env = unify_term(x, y, env)
            if env is None:
                return None
        return env
    return None


def unify_atoms(a1: Atom, a2: Atom) -> Optional[Dict[str, object]]:
    if a1.pred != a2.pred or len(a1.args) != len(a2.args):
        return None
    env: Dict[str, object] = {}
    for x, y in zip(a1.args, a2.args):
        env = unify_term(x, y, env)
        if env is None:
            return None
    return env


def _subst_atom(a: Atom, env) -> Atom:
    return Atom(a.pred, tuple(_apply(t, env) for t in a.args), a.sign)


def _is_tautology(cl: FrozenSet[Atom]) -> bool:
    pos = set()
    neg = set()
    for a in cl:
        if a.sign:
            if (a.pred, a.args) in neg:
                return True
            pos.add((a.pred, a.args))
        else:
            if (a.pred, a.args) in pos:
                return True
            neg.add((a.pred, a.args))
    return False


def _subsumes(c1: FrozenSet[Atom], c2: FrozenSet[Atom]) -> bool:
    if len(c1) >= len(c2):
        return False
    # c1 subsumes c2 if some substitution makes c1 subset of c2
    l1 = list(c1)
    # try all matchings (clauses are small)
    def match(i, env, used):
        if i == len(l1):
            return True
        for j, b in enumerate(list(c2)):
            if j in used or l1[i].sign != b.sign or l1[i].pred != b.pred:
                continue
            e2 = unify_atoms(l1[i], Atom(b.pred, b.args, True))
            # unify signs: use raw args
            e2 = unify_lists(l1[i].args, b.args, dict(env))
            if e2 is not None:
                used.add(j)
                if match(i + 1, e2, used):
                    return True
                used.discard(j)
        return False
    return match(0, {}, set())


def unify_lists(args1, args2, env):
    if len(args1) != len(args2):
        return None
    for x, y in zip(args1, args2):
        env = unify_term(x, y, env)
        if env is None:
            return None
    return env


def _resolve(c1: FrozenSet[Atom], c2: FrozenSet[Atom]) -> List[FrozenSet[Atom]]:
    out = []
    for a in c1:
        for b in c2:
            if a.pred != b.pred or a.sign == b.sign:
                continue
            env = unify_lists(a.args, b.args, {})
            if env is None:
                continue
            rest = set()
            for l in c1:
                if l is not a:
                    rest.add(_subst_atom(l, env))
            for l in c2:
                if l is not b:
                    rest.add(_subst_atom(l, env))
            out.append(frozenset(rest))
    return out


def _factor(cl: FrozenSet[Atom]) -> List[FrozenSet[Atom]]:
    out = []
    lits = list(cl)
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            if lits[i].sign != lits[j].sign or lits[i].pred != lits[j].pred:
                continue
            env = unify_lists(lits[i].args, lits[j].args, {})
            if env is None:
                continue
            rest = set()
            for k, l in enumerate(lits):
                if k != j:
                    rest.add(_subst_atom(l, env))
            out.append(frozenset(rest))
    return out


def prove(premise_clauses: List[FrozenSet[Atom]],
          goal_clauses: List[FrozenSet[Atom]],
          max_clauses: int = 1500,
          time_limit: float = 3.0) -> Tuple[bool, int]:
    """Resolution refutation with set-of-support (SOS) + unit preference.

    SOS starts as the negated-goal clauses; every resolvent must have at
    least one SOS parent.  SOS is complete for refutation and focuses
    search on the goal.  Factoring, tautology deletion and forward
    subsumption keep the clause set small.
    Returns (proved, n_generated).
    """
    t0 = time.time()
    usable: Set[FrozenSet[Atom]] = set()
    for c in premise_clauses:
        if not _is_tautology(c) and not any(
                _subsumes(e, c) for e in usable):
            usable = {e for e in usable if not _subsumes(c, e)}
            usable.add(c)
    sos: List[FrozenSet[Atom]] = []
    sos_set: Set[FrozenSet[Atom]] = set()

    def sos_add(c: FrozenSet[Atom]) -> bool:
        if c in usable or c in sos_set or _is_tautology(c):
            return False
        if any(_subsumes(e, c) for e in usable) or any(
                _subsumes(e, c) for e in sos_set):
            return False
        # backward subsumption inside SOS
        sos_set.add(c)
        sos.append(c)
        return True

    for c in goal_clauses:
        if not c:
            return True, 0
        sos_add(c)
    if not sos:
        return False, 0
    generated = 0
    done: Set[FrozenSet[Atom]] = set()  # SOS clauses fully expanded

    def exhausted() -> bool:
        return (len(usable) + len(sos_set) > max_clauses
                or time.time() - t0 > time_limit)

    while True:
        # unit preference: expand smallest unexpanded SOS clause
        cands = [c for c in sos if c not in done]
        if not cands:
            return False, generated
        given = min(cands, key=lambda c: (len(c), sorted(map(str, c))))
        done.add(given)
        partners = list(usable) + [c for c in sos if c is not given]
        fresh: List[FrozenSet[Atom]] = []
        for p in partners:
            for r in _resolve(given, p):
                generated += 1
                if not r:
                    return True, generated
                fresh.append(r)
                if generated % 256 == 0 and exhausted():
                    return False, generated
        for f in _factor(given):
            generated += 1
            if not f:
                return True, generated
            fresh.append(f)
        for r in fresh:
            if r not in usable and r not in sos_set:
                sos_add(r)
        usable.add(given)
        if exhausted():
            return False, generated



def _apply_ground_equality(clauses: List[FrozenSet[Atom]]) -> List[FrozenSet[Atom]]:
    """Replace c1 by c2 for ground Eq(c1, c2) unit clauses; drop them."""
    parent = {}

    def find(x):
        while parent.get(x, x) != x:
            x = parent[x]
        return x

    rest = []
    for c in clauses:
        if len(c) == 1:
            a = next(iter(c))
            if a.pred == "Eq" and a.sign and len(a.args) == 2:
                from folio_cnf import Const as _C
                x, y = a.args
                if isinstance(x, _C) and isinstance(y, _C) and x.name != y.name:
                    rx, ry = find(x.name), find(y.name)
                    if rx != ry:
                        parent[rx] = ry
                    continue
        rest.append(c)
    if not parent:
        return clauses

    def rep(t):
        from folio_cnf import Const as _C, Func as _F
        if isinstance(t, _C):
            return _C(find(t.name))
        if isinstance(t, _F):
            return _F(t.name, tuple(rep(a) for a in t.args))
        return t

    out = []
    for c in rest:
        nc = set()
        for a in c:
            if a.pred == "Eq":
                x, y = rep(a.args[0]), rep(a.args[1])
                if x == y:
                    if not a.sign:
                        nc = None  # ~(c=c) is unsatisfiable marker; keep clause
                        break
                    else:
                        continue  # (c=c) trivially true; drop literal
                nc.add(Atom("Eq", (x, y), a.sign))
            else:
                nc.add(Atom(a.pred, tuple(rep(t) for t in a.args), a.sign))
        if nc is not None:
            out.append(frozenset(nc))
    return out


def negate_to_cnf(goal_clauses: List[FrozenSet[Atom]]) -> List[FrozenSet[Atom]]:
    """Negate a goal given in CNF: ~(C1 & ... & Cn) = ~C1 | ... | ~Cn.

    Each ~Ci (negation of a disjunction) becomes a conjunction of unit
    complementary literals; distributing the big disjunction yields one
    clause per combination (goals are small in practice).
    """
    options = []
    for c in goal_clauses:
        options.append([Atom(a.pred, a.args, not a.sign) for a in c])
    out = [frozenset()]
    for opt in options:
        nxt = []
        for base in out:
            for lit in opt:
                nxt.append(base | {lit})
        out = nxt
    return out


def answer_fol(premises_fol: List[str], conclusion_fol: str,
               max_clauses: int = 1500,
               time_limit: float = 3.0) -> Tuple[bool, dict]:
    """Prove-or-abstain. Returns (predicted_true, info)."""
    info: dict = {}
    try:
        prem_clauses: List[FrozenSet[Atom]] = []
        for i, p in enumerate(premises_fol):
            p = p.strip()
            if p:
                prem_clauses.extend(to_cnf(p, "_p%d" % i))
        # negate the goal BEFORE Skolemization: goal = AND of lines,
        # refutation needs CNF(~goal)
        goal_asts = []
        for j, g in enumerate(conclusion_fol.split("\n")):
            g = g.strip()
            if g:
                goal_asts.append(parse_formula(g))
        if not goal_asts:
            info["parse_error"] = "empty goal"
            return False, info
        goal_and = goal_asts[0]
        for g in goal_asts[1:]:
            goal_and = ("and", goal_and, g)
        neg_goal = to_cnf_ast(("not", goal_and), "_g")
        goal_clauses = to_cnf_ast(goal_and, "_gpos")
    except ParseError as e:
        info["parse_error"] = str(e)
        return False, info
    prem_clauses = _apply_ground_equality(prem_clauses)
    proved, ngen = prove(prem_clauses, neg_goal, max_clauses, time_limit)
    info["n_prem_clauses"] = len(prem_clauses)
    info["n_goal_clauses"] = len(goal_clauses)
    info["n_generated"] = ngen
    return proved, info

''')

In [7]:
# Part 4 — Unified agent: style router + predict/predict_batch + submission writer.
agent = _mod('agent', r'''"""Unified reasoning agent: NL knowledge base + query -> True/False.

Pipeline (classical AI throughout):
    NL input -> style router -> internal representation ->
    classical reasoning engine -> True/False

Engines:
  * CounterBench-style causal/counterfactual questions ->
    structural causal model + Pearl's 3-step counterfactual procedure
    (abduction by exhaustive root search, do()-interventions,
    forward propagation with fixed-point search).        [counterbench.py]
  * RuleTaker-style fact/rule bases ->
    first-order Horn clauses + forward-chaining least-fixpoint closure
    with negation-as-failure + contradiction retraction. [ruletaker.py]
  * FOLIO-style first-order stories ->
    NL->FOL template parser + resolution refutation prover
    (CNF, unification, factoring, subsumption, set-of-support).
    [folio_nl.py + folio_cnf.py + folio_prover.py]

Any parsing/proving failure abstains -> predicts False (never crashes).
"""
from __future__ import annotations

import re
import time
from typing import Dict, List, Optional, Sequence, Tuple

import counterbench as cb_mod
import ruletaker as rt_mod

_CB_Q = [
    re.compile(r"Would\s+\S+\s+occur\s+if\s+not\s+", re.I),
    re.compile(r"Assume\s+not\s+\S+,?\s+and\s+based\s+on\s+this\s+assumption",
               re.I),
]


def _is_counterbench(context: str, statement: str) -> bool:
    if not re.search(r"\bcauses?\b", context, re.I):
        return False
    return any(p.search(statement) for p in _CB_Q)


def _try_ruletaker(context: str, statement: str):
    kb = rt_mod.parse_context(context)
    q = rt_mod.parse_query(statement)
    return kb, q


def predict(context: str, statement: str) -> Dict:
    """Predict entailment. Returns dict with label/time/engine details."""
    t0 = time.time()
    context = (context or "").strip()
    statement = (statement or "").strip()
    if not context or not statement:
        return {"label": False, "engine": "empty-input",
                "time_s": time.time() - t0, "detail": "empty input"}
    # 1. counterfactual causal style
    if _is_counterbench(context, statement):
        try:
            ans = cb_mod.solve(context, statement, cb_mod.Semantics())
            return {"label": ans == "yes", "engine": "counterbench-scm",
                    "time_s": time.time() - t0, "detail": ans}
        except Exception as e:  # fall through to general engines
            detail = "cb-fallback: %s" % e
        else:
            detail = ""
    # 2. Horn fact/rule style
    try:
        kb = rt_mod.parse_context(context)
        q = rt_mod.parse_query(statement)
        closure = rt_mod.cwa_closure(kb)
        if q[2]:
            label = q in closure.facts
        else:
            label = (q[0], q[1], True) not in closure.facts
        return {"label": label, "engine": "ruletaker-fc-cwa",
                "time_s": time.time() - t0,
                "detail": "closure=%d" % len(closure.facts)}
    except Exception:
        pass
    # 3. general first-order style
    try:
        import folio_nl as nl_mod
        import folio_prover as fp_mod
        prems_nl = [p.strip() for p in re.split(r"\n+", context) if p.strip()]
        if len(prems_nl) <= 1:
            prems_nl = [p.strip() for p in context.split(". ") if p.strip()]
        prems, concl, info = nl_mod.parse_story(prems_nl, statement)
        proved, pinfo = fp_mod.answer_fol(prems, concl)
        return {"label": bool(proved), "engine": "folio-resolution",
                "time_s": time.time() - t0,
                "detail": "proved=%s" % proved}
    except Exception as e:
        return {"label": False, "engine": "abstain",
                "time_s": time.time() - t0, "detail": "error: %s" % e}


def predict_batch(items: Sequence[Tuple[str, str, str]]) -> List[Dict]:
    """Predict for [(id, context, statement), ...].

    Items sharing an identical context are answered together with a
    single parse + single reasoning-closure computation.
    """
    # group row indices by context (preserve global order in output)
    groups: Dict[str, List[int]] = {}
    for i, (_, ctx, _) in enumerate(items):
        groups.setdefault(ctx or "", []).append(i)
    out: List[Optional[Dict]] = [None] * len(items)
    for ctx, idxs in groups.items():
        first = items[idxs[0]]
        try:
            batch = _predict_context(ctx, [items[i][2] for i in idxs])
        except Exception:
            batch = None
        if batch is None:
            for i in idxs:
                out[i] = predict(ctx, items[i][2])
                out[i]["id"] = items[i][0]
            continue
        for i, r in zip(idxs, batch):
            r["id"] = items[i][0]
            out[i] = r
    return [r for r in out if r is not None]


def _predict_context(context: str, statements: List[str]) -> Optional[List[Dict]]:
    """Answer many queries for one shared context (single closure).

    Returns None when the context does not fit the shared-context
    engines (caller falls back to per-item prediction).
    """
    t0 = time.time()
    context = (context or "").strip()
    if not context:
        return None
    # counterfactual style never shares contexts meaningfully; but handle
    # uniformly through the per-item path
    if _is_counterbench(context, statements[0] if statements else ""):
        return None
    # Horn style: one parse + one closure for all queries
    try:
        kb = rt_mod.parse_context(context)
        queries = [rt_mod.parse_query(s) for s in statements]
    except Exception:
        return None
    try:
        closure = rt_mod.cwa_closure(kb)
    except Exception:
        return None
    dt = time.time() - t0
    out = []
    for q in queries:
        if q[2]:
            label = q in closure.facts
        else:
            label = (q[0], q[1], True) not in closure.facts
        out.append({"label": label, "engine": "ruletaker-fc-cwa",
                    "time_s": dt / max(1, len(queries)),
                    "detail": "closure=%d" % len(closure.facts)})
    return out


def write_submission(results: Sequence[Dict], path: str) -> None:
    """Write submission.csv with columns id,predicted_label (True/False)."""
    lines = ["id,predicted_label"]
    for r in results:
        lines.append("%s,%s" % (r["id"], "True" if r["label"] else "False"))
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")

''')

## Smoke tests and reasoning traces
No data files needed. Shows derived facts, the causal equations, and a resolution proof.

In [9]:
"""Smoke tests + reasoning traces (run without any data files)."""
# 1. Assignment example (RuleTaker style)
ctx = ("Bob is young. Bob is kind. "
       "If someone is young, then they are smart. "
       "If someone is smart and kind, then they are successful.")
q = "Bob is successful."
out = agent.predict(ctx, q)
print("assignment example:", q, "->", out["label"], "(expected True)",
      "[engine=%s, %.3fs]" % (out["engine"], out["time_s"]))

import ruletaker as rt
kb = rt.parse_context(ctx)
cl = rt.cwa_closure(kb)
print("closure size:", len(cl.facts))
for lit in sorted(cl.facts, key=rt.render_lit):
    print("  derived:", rt.render_lit(lit))

# 2. CounterBench trace
import counterbench as cbm
gi = "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo."
qq = "Would Lumbo occur if not Ziklo instead of Ziklo?"
model = cbm.parse_context(gi)
eqs = cbm.build_equations(model)
print()
print("counterbench model equations:")
for v, e in eqs.items():
    print("  ", e.render(v))
print("roots:", [v for v in model.variables if v not in eqs],
      "observed:", model.observed)
print("answer:", cbm.solve(gi, qq, cbm.Semantics()), "(expected no)")

# 3. FOLIO resolution trace (gold FOL)
from folio_prover import answer_fol
prems = ["∀x (Human(x) → Mammal(x))", "Human(socrates)"]
pred, info = answer_fol(prems, "Mammal(socrates)")
print()
print("folio demo: Human(socrates), ∀x(Human(x)→Mammal(x)) |= Mammal(socrates)?",
      pred, "(expected True)", info)
pred2, _ = answer_fol(prems, "Mammal(plato)")
print("same premises |= Mammal(plato)?", pred2, "(expected False)")


assignment example: Bob is successful. -> True (expected True) [engine=ruletaker-fc-cwa, 0.001s]
closure size: 4
  derived: Kind(Bob)
  derived: Smart(Bob)
  derived: Successful(Bob)
  derived: Young(Bob)

counterbench model equations:
   Blaf := (Ziklo)
   Trune := (Blaf)
   Vork := (Trune)
   Lumbo := (Vork)
roots: ['Ziklo'] observed: {}
answer: no (expected no)

folio demo: Human(socrates), ∀x(Human(x)→Mammal(x)) |= Mammal(socrates)? True (expected True) {'n_prem_clauses': 2, 'n_goal_clauses': 1, 'n_generated': 2}
same premises |= Mammal(plato)? False (expected False)


## Evaluation (prints final accuracy and inference time)
Uses `./data` if present, else downloads the public Hugging Face sets once. Skips gracefully offline.

In [11]:
"""Evaluation: accuracy + inference time on the public sets.

Data are loaded from ./data if present, otherwise downloaded once from
Hugging Face (urllib only). If neither is available, built-in smoke tests
run and the reference results obtained during development are reported.
"""
import time
import urllib.request
from collections import Counter

DATA = "data"
HAVE = {
    "rt_dev": os.path.exists(os.path.join(DATA, "ruletaker", "dev.parquet")),
    "rt_test": os.path.exists(os.path.join(DATA, "ruletaker", "test.parquet")),
    "cb": os.path.exists(os.path.join(DATA, "counterbench",
                                      "data_balanced_alpha_V1.json")),
    "folio": os.path.exists(os.path.join(DATA, "folio",
                                         "folio_v2_validation.jsonl")),
}

FILES = {
    "ruletaker/dev.parquet": "https://huggingface.co/datasets/tasksource/ruletaker/resolve/main/data/dev-00000-of-00001-480b21836eb3db96.parquet",
    "ruletaker/test.parquet": "https://huggingface.co/datasets/tasksource/ruletaker/resolve/main/data/test-00000-of-00001-402c217dbcd02599.parquet",
    "counterbench/data_balanced_alpha_V1.json": "https://huggingface.co/datasets/CounterBench/CounterBench/resolve/main/data_balanced_alpha_V1.json",
    "counterbench/data_balanced_backdoor_V2.json": "https://huggingface.co/datasets/CounterBench/CounterBench/resolve/main/data_balanced_backdoor_V2.json",
    "folio/folio_v2_train.jsonl": "https://huggingface.co/datasets/tasksource/folio/resolve/main/folio_v2_train.jsonl",
    "folio/folio_v2_validation.jsonl": "https://huggingface.co/datasets/tasksource/folio/resolve/main/folio_v2_validation.jsonl",
}


def ensure_data():
    for rel, url in FILES.items():
        p = os.path.join(DATA, rel)
        if not os.path.exists(p):
            try:
                os.makedirs(os.path.dirname(p), exist_ok=True)
                print("downloading", rel)
                urllib.request.urlretrieve(url, p)
            except Exception as e:
                print("download failed:", rel, e)


try:
    import pandas as pd
    _HAVE_PD = True
except Exception:
    _HAVE_PD = False
    print("pandas not available: RuleTaker file evaluation skipped")

if not all(HAVE.values()):
    ensure_data()

results = {}


def eval_ruletaker(split):
    import pandas as pd
    df = pd.read_parquet(os.path.join(DATA, "ruletaker", "%s.parquet" % split))
    correct = n = 0
    t0 = time.time()
    for ctx, group in df.groupby("context", sort=False):
        preds = agent.predict_batch(
            [(str(i), ctx, q) for i, q in enumerate(group.question.tolist())])
        labels = (group.label == "entailment").tolist()
        for r, l in zip(preds, labels):
            n += 1
            correct += (r["label"] == l)
    dt = time.time() - t0
    print("RuleTaker %s: %d/%d = %.2f%%  (%.1fs, %.0f q/s)" %
          (split, correct, n, 100.0 * correct / n, dt, n / dt))
    return correct / n, dt


def eval_counterbench():
    import json as _json
    rows = []
    for f in ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]:
        rows += _json.load(open(os.path.join(DATA, "counterbench", f),
                                encoding="utf-8"))
    correct = 0
    t0 = time.time()
    for r in rows:
        out = agent.predict(r["given_info"], r["question"])
        if out["label"] == (r["answer"] == "yes"):
            correct += 1
    dt = time.time() - t0
    print("CounterBench: %d/%d = %.2f%%  (%.1fs, %.0f q/s)" %
          (correct, len(rows), 100.0 * correct / len(rows), dt,
           len(rows) / dt))
    return correct / len(rows), dt


def eval_folio(which="validation", use_gold_fol=False):
    import json as _json
    path = os.path.join(DATA, "folio", "folio_v2_%s.jsonl" % which)
    rows = [_json.loads(l) for l in open(path, encoding="utf-8")]
    correct = 0
    t0 = time.time()
    for r in rows:
        if use_gold_fol:
            from folio_prover import answer_fol
            pred, _ = answer_fol(r["premises-FOL"].split("\n"),
                                r["conclusion-FOL"])
        else:
            ctx = r["premises"]
            out = agent.predict(ctx, r["conclusion"])
            pred = out["label"]
        if pred == (r["label"] == "True"):
            correct += 1
    dt = time.time() - t0
    tag = "FOLIO-%s[%s]" % (which, "gold-FOL" if use_gold_fol else "NL-only")
    print("%s: %d/%d = %.2f%%  (%.1fs, %.1f q/s)" %
          (tag, correct, len(rows), 100.0 * correct / len(rows), dt,
           len(rows) / dt))
    return correct / len(rows), dt


if _HAVE_PD and os.path.exists(os.path.join(DATA, "ruletaker", "dev.parquet")):
    results["ruletaker-dev"] = eval_ruletaker("dev")
    results["ruletaker-test"] = eval_ruletaker("test")
if os.path.exists(os.path.join(DATA, "counterbench",
                               "data_balanced_alpha_V1.json")):
    results["counterbench"] = eval_counterbench()
if os.path.exists(os.path.join(DATA, "folio", "folio_v2_validation.jsonl")):
    results["folio-val-goldFOL"] = eval_folio("validation", use_gold_fol=True)
    results["folio-val-NL"] = eval_folio("validation", use_gold_fol=False)
    results["folio-train-NL"] = eval_folio("train", use_gold_fol=False)

print()
print("FINAL accuracy / inference time:")
total_c = total_n = 0.0
total_t = 0.0
for k, (acc, dt) in results.items():
    print("  %-22s acc=%.2f%% time=%.1fs" % (k, 100 * acc, dt))
    total_t += dt
print("total inference time: %.1fs" % total_t)


RuleTaker dev: 75872/75872 = 100.00%  (11.6s, 6551 q/s)
RuleTaker test: 151911/151911 = 100.00%  (32.6s, 4666 q/s)
CounterBench: 1140/1200 = 95.00%  (0.5s, 2376 q/s)
FOLIO-validation[gold-FOL]: 165/203 = 81.28%  (20.7s, 9.8 q/s)
FOLIO-validation[NL-only]: 132/203 = 65.02%  (0.7s, 284.7 q/s)
FOLIO-train[NL-only]: 607/1001 = 60.64%  (3.9s, 255.4 q/s)

FINAL accuracy / inference time:
  ruletaker-dev          acc=100.00% time=11.6s
  ruletaker-test         acc=100.00% time=32.6s
  counterbench           acc=95.00% time=0.5s
  folio-val-goldFOL      acc=81.28% time=20.7s
  folio-val-NL           acc=65.02% time=0.7s
  folio-train-NL         acc=60.64% time=3.9s
total inference time: 70.0s


## submission.csv
Predicts a `private_test.csv`/`test.csv` file when present (columns auto-detected), else writes a demo-format file from public samples. Output columns: `id,predicted_label`.

In [13]:
"""Generate submission.csv.

If a private/test file is present (private_test.csv, test.csv or
data/private_test.csv with columns id + context/question inputs), it is
used. Otherwise a demonstration submission.csv is generated from a mixed
sample of the public sets (same two-column format: id,predicted_label).
"""
import csv
import json as _json

CANDIDATES = ["private_test.csv", "test.csv", "data/private_test.csv",
              "data/test.csv"]
src = next((p for p in CANDIDATES if os.path.exists(p)), None)


def _pick(row, options):
    for c in options:
        if c in row and row[c] not in (None, ""):
            return row[c]
    for c in options:
        for k in row:
            if k.lower() == c.lower() and row[k] not in (None, ""):
                return row[k]
    return None


items = []
if src is not None:
    print("using test file:", src)
    with open(src, encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            rid = _pick(row, ["id", "question_id", "example_id", "index"])
            ctx = _pick(row, ["context", "premises", "given_info", "kb",
                              "theory", "text"])
            stmt = _pick(row, ["statement", "question", "query", "conclusion",
                               "claim"])
            if rid is None or stmt is None:
                continue
            # counterbench-style rows carry question+given_info separately
            items.append((str(rid), str(ctx or ""), str(stmt)))
else:
    print("no private test file found: building demo submission sample")
    import pandas as pd
    df = pd.read_parquet(os.path.join(DATA, "ruletaker", "test.parquet"))
    demo = df.sample(n=min(1000, len(df)), random_state=0)
    for i, row in enumerate(demo.itertuples()):
        items.append(("demo-rt-%04d" % i, row.context, row.question))
    cbrows = _json.load(open(os.path.join(
        DATA, "counterbench", "data_balanced_alpha_V1.json"),
        encoding="utf-8"))[:500]
    for i, r in enumerate(cbrows):
        items.append(("demo-cb-%04d" % i, r["given_info"], r["question"]))
    frows = [_json.loads(l) for l in open(
        os.path.join(DATA, "folio", "folio_v2_validation.jsonl"),
        encoding="utf-8")][:203]
    for i, r in enumerate(frows):
        items.append(("demo-folio-%04d" % i, r["premises"], r["conclusion"]))

t0 = time.time()
results = agent.predict_batch(items)
dt = time.time() - t0
agent.write_submission(results, "submission.csv")
n_true = sum(1 for r in results if r["label"])
print("wrote submission.csv: %d rows (%d True, %d False) in %.1fs" %
      (len(results), n_true, len(results) - n_true, dt))
print("engines used:", Counter(r["engine"] for r in results))


no private test file found: building demo submission sample
wrote submission.csv: 1703 rows (771 True, 932 False) in 2.4s
engines used: Counter({'ruletaker-fc-cwa': 1000, 'counterbench-scm': 500, 'folio-resolution': 203})
